# Aula — Classificação com scikit-learn e Métricas de Classificação

**Métodos e Aplicações de Inteligência Artificial (IBM3116) · Engenharia da Computação · Ibmec São Paulo**

Na aula anterior você implementou a regressão logística do zero. Agora vamos usar o **scikit-learn** para construir classificadores de forma profissional e, principalmente, responder a uma pergunta que decide se um sistema de classificação serve ou não para alguma coisa:

> **Como saber se um classificador é bom — e bom para quê?**

Vamos estudar a **matriz de confusão**, **acurácia**, **precisão**, **recall**, **especificidade**, **F1**, o papel do **limiar de decisão**, a **curva ROC**, a **AUC** e a **curva precisão–recall**. Para cada métrica veremos o que ela mede, o que ela esconde e quando usá-la.

O caso de estudo é real e vem da Engenharia de Software: **prever quais módulos de um sistema da NASA contêm defeitos** a partir de métricas estáticas do código-fonte.

**Como usar este notebook.** Leia as explicações, execute as células em ordem e resolva as tarefas.

| Tarefa | Tema |
|---|---|
| Exercício 1 | Da matriz de confusão às métricas |
| Exercício 2 |  Detectores ingênuos e a regra de McCabe |
| Exercício 3 |  Varredura de limiares com restrição operacional |
| Exercício 4 |  ROC e AUC do zero — e um detector de uma variável só |
| Investigação final | Política de inspeção de código para uma release |

---
## 1. Por que estudar isso?

Imagine que você é engenheiro de qualidade de um sistema de solo que recebe e processa dados de satélites. O sistema tem cerca de **2.000 módulos** (funções e métodos em C++). Antes de cada *release*, a equipe pode fazer revisão de código e testes adicionais, mas essas atividades consomem horas de engenheiros sêniores: **não dá para inspecionar tudo**. A pergunta prática é:

> **Quais módulos devemos inspecionar primeiro?**

Uma abordagem clássica é usar **métricas estáticas de código** (tamanho, complexidade ciclomática, contagem de operadores etc.), que podem ser extraídas automaticamente, para estimar quais módulos têm maior chance de conter defeitos (Menzies, Greenwald & Frank, 2007). Isso é um **problema de classificação binária**: para cada módulo, decidir entre "inspecionar" (provável defeito) e "não inspecionar".

Agora, uma provocação. Um colega apresenta um classificador com **84,5% de acurácia**. Parece bom? O "modelo" dele é:

```python
def classificador_do_colega(modulo):
    return 0   # "este módulo não tem defeito"
```

Como 84,5% dos módulos do conjunto que usaremos não têm defeito registrado, essa regra acerta 84,5% das vezes — e **não encontra nenhum defeito**. A própria documentação do conjunto de dados registra essa lição: *"Accuracy is a surprisingly uninformative measure of success for a defect detector"* (cabeçalho do arquivo KC1 do PROMISE Repository).

Esse exemplo resume tudo isso aqui: **a métrica precisa refletir o objetivo do sistema**. Ao longo do notebook responderemos:

1. Quais são os tipos de erro de um classificador e quanto custa cada um?
2. Como medir cada tipo de erro separadamente?
3. Como escolher o **ponto de operação** (limiar) de acordo com a capacidade da equipe ou com os custos?
4. Como resumir a qualidade de um classificador **independentemente do limiar**?

O mesmo raciocínio aparece em muitos problemas de Engenharia da Computação com **classes raras**: detecção de intrusão em redes, detecção de falhas em sensores e sistemas embarcados, manutenção preditiva, detecção de fraude e filtragem de spam.

---
## 3. Onde este tema está dentro da Inteligência Artificial?

Estamos em **aprendizado supervisionado**, na tarefa de **classificação**. Nas aulas de regressão, o alvo era contínuo ($y \in \mathbb{R}$) e avaliávamos o modelo com MSE, RMSE e $R^2$. Agora o alvo é uma categoria; nesta aula, binária: $y \in \{0, 1\}$.

A maioria dos classificadores modernos não produz a classe diretamente. Eles produzem um **escore** (uma probabilidade estimada ou um logit), e uma **regra de decisão** transforma esse escore em classe:

```
x (21 métricas do módulo) ──► modelo f(x; θ) ──► escore s(x) ──► regra ŷ = 𝕀[s(x) ≥ t] ──► classe ŷ ∈ {0, 1}
                              parâmetros θ                         limiar t
                              ajustados no treino                  decisão de engenharia
```

Essa separação organiza as métricas em dois grupos:

| Avaliam a **classe** $\hat y$ (dependem do limiar $t$) | Avaliam o **escore** $s(\mathbf{x})$ (não dependem de $t$) |
|---|---|
| matriz de confusão, acurácia, precisão, recall, especificidade, F1 | curva ROC, AUC, curva precisão–recall, *average precision* |

Dois pontos importantes:

- As métricas desta aula são **agnósticas ao modelo**: servem para a regressão logística de hoje, para KNN e árvores de decisão (próxima aula), para *ensembles* e para redes neurais.
- Um **sistema baseado em regras** também é um classificador. A regra "inspecionar se a complexidade ciclomática for maior que 10" pode ser avaliada exatamente com as mesmas métricas que um modelo aprendido — e faremos isso no Exercício 2.

---
## 4. Vamos lembrar de algumas coisas!

### 4.1 Regressão logística (aula anterior)

Para um módulo descrito pelo vetor de atributos $\mathbf{x} \in \mathbb{R}^{p}$ (aqui, $p = 21$):

$$
z = \mathbf{w}^\top \mathbf{x} + b, \qquad \hat p(\mathbf{x}) = \sigma(z) = \frac{1}{1 + e^{-z}},
$$

em que $\mathbf{w} \in \mathbb{R}^{p}$ são os pesos, $b \in \mathbb{R}$ é o viés (intercepto), $z \in \mathbb{R}$ é o **logit** e $\hat p(\mathbf{x}) \in (0, 1)$ é a estimativa de $P(y = 1 \mid \mathbf{x})$. Os parâmetros são ajustados minimizando a **entropia cruzada binária** média sobre os $n$ exemplos de treino:

$$
J(\mathbf{w}, b) = -\frac{1}{n} \sum_{i=1}^{n} \Big[ y_i \log \hat p(\mathbf{x}_i) + (1 - y_i) \log\big(1 - \hat p(\mathbf{x}_i)\big) \Big].
$$

*Procedência:* formulação da documentação do scikit-learn, seção *Logistic regression* do guia do usuário (scikit-learn, 2026), a mesma usada na aula anterior. Na Seção 7 veremos a versão com regularização que o scikit-learn realmente otimiza.

### 4.2 Probabilidade condicional e Teorema de Bayes

$$
P(A \mid B) = \frac{P(A \cap B)}{P(B)}, \qquad P(A \mid B) = \frac{P(B \mid A)\,P(A)}{P(B)}.
$$

*Procedência:* Russell & Norvig (2022), cap. 12. Guarde a ideia de que $P(A \mid B) \neq P(B \mid A)$: veremos que **precisão** e **recall** são exatamente duas probabilidades condicionais com a condição trocada.

### 4.3 Média harmônica

Para $a, b > 0$:

$$
H(a, b) = \frac{2}{\frac{1}{a} + \frac{1}{b}} = \frac{2ab}{a + b}.
$$

A média harmônica fica próxima do **menor** dos dois valores: para $a = 1{,}0$ e $b = 0{,}1$, a média aritmética é $0{,}55$, mas $H = 0{,}18$. É isso que torna o F1 exigente.

### 4.4 Área sob uma curva poligonal (regra do trapézio)

Para pontos $(x_0, y_0), \dots, (x_m, y_m)$ ordenados por $x$, a área sob a poligonal que os liga é

$$
A = \sum_{k=1}^{m} (x_k - x_{k-1}) \, \frac{y_k + y_{k-1}}{2}.
$$

*Procedência:* regra do trapézio; é o procedimento usado para calcular a AUC em Fawcett (2006, Algoritmo 2).

---
## 5. Preparando o ambiente

Todas as bibliotecas usadas fazem parte do ambiente `ia2026` configurado no início da disciplina. Fixamos uma semente (`SEED`) para que as divisões dos dados sejam reprodutíveis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    balanced_accuracy_score, roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
)

SEED = 42

# Aparência dos gráficos
plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False,
})
COR_LR = "#2a78d6"     # regressão logística / classe 0
COR_ALT = "#eb6834"    # modelo alternativo / classe 1 (defeito)
COR_3 = "#1baf7a"      # terceira série
COR_BASE = "#7a7a7a"   # linhas de base (baselines)

pd.set_option("display.precision", 3)
print(f"NumPy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")

---
# Parte I — O problema e os dados

## 6. O conjunto de dados KC1 (NASA Metrics Data Program / PROMISE)

**KC1** é um sistema em C++ da NASA que implementa gerenciamento de armazenamento para recepção e processamento de dados de solo. Cada linha do conjunto é um **módulo** — a menor unidade de funcionalidade (uma função em C, um método em C++). Para cada módulo foram extraídas **21 métricas estáticas** do código-fonte, e o alvo `defects` indica se o módulo teve **um ou mais defeitos reportados**.

Os dados foram publicados pelo NASA Metrics Data Program, disponibilizados no PROMISE Repository (Sayyad Shirabad & Menzies, 2005) e estão hospedados no OpenML (Vanschoren et al., 2014) com identificador `data_id=1067`. O download é feito diretamente pelo scikit-learn, sem cadastro.

### 6.1 O que são os atributos?

**Métricas de McCabe (McCabe, 1976).** A **complexidade ciclomática** de um módulo é calculada sobre o grafo de fluxo de controle $G$:

$$
v(G) = e - n + 2,
$$

em que $e$ é o número de arestas (desvios de fluxo) e $n$ é o número de nós (comandos). Ela conta os **caminhos linearmente independentes** do módulo. Para um módulo estruturado com uma entrada e uma saída, $v(G)$ é igual ao número de decisões (predicados) mais 1 (McCabe, 1976). Exemplo: uma função com um `if` e um `while` tem $v(G) = 3$.

**Métricas de Halstead (Halstead, 1977).** Contam "conceitos" no código: $\eta_1$ operadores distintos, $\eta_2$ operandos distintos, $N_1$ ocorrências de operadores e $N_2$ ocorrências de operandos. Delas derivam, entre outras:

$$
N = N_1 + N_2, \qquad \eta = \eta_1 + \eta_2, \qquad V = N \log_2 \eta,
$$

em que $N$ é o comprimento do programa e $V$ é o **volume**. *Procedência:* Halstead (1977), conforme documentado no cabeçalho do arquivo KC1.

| Coluna | Significado |
|---|---|
| `loc` | linhas de código (convenção de contagem de McCabe) |
| `v(g)`, `ev(g)`, `iv(g)` | complexidade ciclomática, essencial e de projeto (McCabe) |
| `n`, `v` | comprimento $N$ e volume $V$ (Halstead) |
| `l`, `d`, `i`, `e`, `t` | nível, dificuldade, "inteligência", esforço e tempo estimado (Halstead) |
| `b` | estimativa de Halstead associada a erros (o arquivo não detalha a fórmula) |
| `lOCode`, `lOComment`, `lOBlank`, `locCodeAndComment` | contagens de linhas de código, comentário, em branco e mistas |
| `uniq_Op`, `uniq_Opnd`, `total_Op`, `total_Opnd` | $\eta_1$, $\eta_2$, $N_1$, $N_2$ |
| `branchCount` | número de ramos no grafo de fluxo |
| `defects` | **alvo**: `true` se o módulo teve defeito reportado |

> **Por que isso é aprendizado de máquina e não apenas uma regra?** McCabe e Halstead propuseram essas métricas nos anos 1970 como indicadores de código propenso a erros, mas a relação entre métrica e defeito é estatística, não determinística: dois módulos com a mesma complexidade podem ter destinos diferentes. Um classificador estima essa relação a partir de dados históricos.

In [ ]:
# Download do OpenML (≈170 kB). Na primeira execução o arquivo fica em cache em
# ~/scikit_learn_data; nas execuções seguintes ele é lido do disco.
try:
    kc1 = fetch_openml(data_id=1067, as_frame=True, parser="auto")
except Exception as erro:
    raise RuntimeError(
        "Não foi possível baixar o KC1 do OpenML. Verifique a conexão com a internet "
        "(o endereço openml.org precisa estar acessível) e execute a célula novamente."
    ) from erro

df = kc1.frame.copy()
X = df.drop(columns="defects")                 # matriz de atributos (n × p)
y = (df["defects"] == "true").astype(int)      # alvo: 1 = módulo com defeito

print(f"n = {X.shape[0]} módulos, p = {X.shape[1]} atributos")
df.head()

### 6.2 Quão raros são os defeitos?

A fração de exemplos positivos é a **prevalência** da classe positiva:

$$
\pi = \frac{1}{n}\sum_{i=1}^{n} y_i = \frac{P}{P + N},
$$

em que $P$ é o número de módulos com defeito e $N$ o número de módulos sem defeito. A prevalência é a primeira coisa a olhar em um problema de classificação: ela determina o desempenho dos classificadores triviais e o valor de várias métricas.

In [ ]:
contagem = y.value_counts().sort_index()
prevalencia = y.mean()

tabela_classes = pd.DataFrame({
    "módulos": contagem,
    "proporção": contagem / len(y),
}, index=pd.Index([0, 1], name="defects"))
display(tabela_classes)
print(f"Prevalência de defeitos: π = {prevalencia:.4f}")

Apenas **326 de 2.109 módulos (15,5%)** têm defeito. Dizemos que o problema é **desbalanceado**. Guarde o número $1 - \pi \approx 0{,}845$: é a acurácia do classificador do colega da Seção 1.

### 6.3 Como os atributos se distribuem?

In [ ]:
resumo = X.describe().T[["mean", "50%", "max"]]
resumo.columns = ["média", "mediana", "máximo"]
resumo["assimetria"] = X.skew()
resumo

In [ ]:
# Distribuição de linhas de código (escala log) para cada classe
bins = np.logspace(0, np.log10(X["loc"].max() + 1), 30)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.hist(X.loc[y == 0, "loc"] + 1, bins=bins, alpha=0.6, color=COR_LR, label="sem defeito (y = 0)", density=True)
ax.hist(X.loc[y == 1, "loc"] + 1, bins=bins, alpha=0.6, color=COR_ALT, label="com defeito (y = 1)", density=True)
ax.set_xscale("log")
ax.set_xlabel("loc + 1 (escala logarítmica)")
ax.set_ylabel("densidade")
ax.set_title("Módulos com defeito tendem a ser maiores, mas as distribuições se sobrepõem")
ax.legend()
plt.show()

### O que acabamos de observar?

1. **Escalas muito diferentes e caudas longas.** O esforço de Halstead `e` tem média ≈ 5.242, mas mediana ≈ 214 e máximo ≈ 324.804. Quase todos os atributos têm assimetria positiva forte (média ≫ mediana). Isso tem duas consequências práticas:
   - a **padronização** é necessária: a penalização L2 da regressão logística trata todos os pesos igualmente, e o otimizador converge mal quando os atributos têm escalas de ordens de grandeza distintas;
   - uma **transformação logarítmica** $x' = \log(1 + x)$ comprime as caudas. Menzies, Greenwald & Frank (2007) aplicaram um filtro logarítmico aos atributos numéricos de conjuntos de dados da NASA MDP e relataram melhora dos preditores. No scikit-learn, isso se faz com `FunctionTransformer(np.log1p)` dentro de um `Pipeline` — você poderá testar essa opção na investigação final.

2. **Sobreposição das classes.** Módulos com defeito tendem a ser maiores, mas há muitos módulos grandes sem defeito e módulos pequenos com defeito. Nenhum limiar em `loc` separa as classes perfeitamente. **Classes sobrepostas implicam erros inevitáveis**; a questão passa a ser *quais* erros aceitar.

### 6.4 Qualidade dos dados: módulos repetidos

Antes de modelar, vale verificar duplicatas. Shepperd et al. (2013) mostraram que os conjuntos da NASA MDP contêm problemas de qualidade, entre eles casos repetidos e casos idênticos com rótulos diferentes.

In [ ]:
n_repetidas = X.duplicated().sum()     # linhas cujos 21 atributos repetem uma linha anterior
rotulos_por_vetor = df.groupby(list(X.columns), sort=False)["defects"].nunique()
conflitantes = rotulos_por_vetor[rotulos_por_vetor > 1]
linhas_envolvidas = X.duplicated(keep=False)

print(f"Vetores de atributos distintos:                      {len(rotulos_por_vetor)} de {len(X)} linhas")
print(f"Linhas que repetem os atributos de outra linha:       {n_repetidas}")
print(f"Vetores idênticos com rótulos diferentes (conflitos): {len(conflitantes)}")
print(f"Prevalência entre linhas repetidas:     {y[linhas_envolvidas].mean():.3f}")
print(f"Prevalência entre linhas não repetidas: {y[~linhas_envolvidas].mean():.3f}")
print(f"Mediana de loc (repetidas / não repetidas): {X.loc[linhas_envolvidas, 'loc'].median():.0f} / {X.loc[~linhas_envolvidas, 'loc'].median():.0f}")

### O que acabamos de observar?

- Há apenas **1.192 vetores de atributos distintos** entre 2.109 linhas. As repetições são, sobretudo, **módulos minúsculos** (mediana de 4 linhas) e quase sempre sem defeito — pense em *getters*, *setters* e funções triviais, que produzem exatamente as mesmas métricas.
- Existem **vetores idênticos com rótulos diferentes**. Para esses casos, **nenhuma** função $f(\mathbf{x})$ acerta todos os exemplos: há um piso de erro irredutível com esses atributos.
- Em uma divisão aleatória, cópias idênticas podem cair no treino e no teste. Modelos que **memorizam** exemplos (como KNN com $k=1$ ou árvores muito profundas, da próxima aula) podem ter métricas infladas por isso. A regressão logística é pouco sensível a esse efeito, mas a **população avaliada** continua cheia de módulos triviais e fáceis de classificar.

Nesta aula usaremos os dados como são distribuídos (prática comum na literatura), mas **registre esta limitação**: ela deve aparecer na conclusão da investigação final. Estratégias de divisão que respeitam grupos serão tratadas na aula de avaliação rigorosa de modelos.

---
# Parte II — Classificação com scikit-learn

## 7. A API de estimadores do scikit-learn

O scikit-learn organiza todos os algoritmos com a mesma interface (Buitinck et al., 2013). Entender essa interface é o que permite trocar a regressão logística por KNN, árvores ou *ensembles* nas próximas aulas mudando uma linha.

| Conceito | O que é | Exemplo |
|---|---|---|
| **Estimador** | objeto configurado por **hiperparâmetros** passados no construtor. Hiperparâmetros **não** são ajustados pelos dados. | `LogisticRegression(C=1.0, max_iter=1000)` |
| `fit(X, y)` | ajusta os **parâmetros** usando os dados de treino. Parâmetros aprendidos terminam com `_`. | `coef_`, `intercept_`, `classes_` |
| **Transformador** | estimador com `fit` + `transform`. | `StandardScaler` aprende `mean_` e `scale_` |
| **Classificador** | estimador com `predict` e, em geral, `predict_proba` e/ou `decision_function`. | `LogisticRegression`, `DummyClassifier` |
| **Pipeline** | encadeia transformadores e um estimador final. `fit` ajusta cada etapa **só com os dados de treino**; `predict` aplica as transformações já ajustadas. | `make_pipeline(StandardScaler(), LogisticRegression())` |

### 7.1 Da matemática à API

A padronização do `StandardScaler` calcula, para cada atributo $j$, a média $\mu_j$ e o desvio-padrão $s_j$ **no treino** e aplica $\tilde x_j = (x_j - \mu_j)/s_j$ (scikit-learn, 2026, `StandardScaler`). A regressão logística então usa os atributos padronizados $\tilde{\mathbf{x}}$:

| Matemática | Dimensão | scikit-learn |
|---|---|---|
| $\mu_j,\ s_j$ | $p$ cada | `scaler.mean_`, `scaler.scale_` |
| $\mathbf{w},\ b$ | $1 \times p$ e $1$ | `logreg.coef_`, `logreg.intercept_` |
| $z = \mathbf{w}^\top \tilde{\mathbf{x}} + b$ | $n$ escores | `decision_function(X)` |
| $\big[1 - \hat p(\mathbf{x}),\ \hat p(\mathbf{x})\big]$ | $n \times 2$ | `predict_proba(X)` (colunas na ordem de `classes_`) |
| $\hat y = \mathbb{I}[z > 0] = \mathbb{I}[\hat p(\mathbf{x}) > 0{,}5]$ | $n$ classes | `predict(X)` |

### 7.2 O que `fit` otimiza na `LogisticRegression`?

Com a configuração padrão (penalização L2), o scikit-learn resolve

$$
\min_{\mathbf{w},\, b} \;\; \frac{1}{S} \sum_{i=1}^{n} s_i \, \ell_i(\mathbf{w}, b) \;+\; \frac{1}{S\,C} \cdot \frac{1}{2}\lVert \mathbf{w} \rVert_2^2,
\qquad
\ell_i = -y_i \log \hat p(\mathbf{x}_i) - (1 - y_i)\log\big(1 - \hat p(\mathbf{x}_i)\big),
$$

em que:

- $\ell_i$ é a entropia cruzada binária do exemplo $i$ (a mesma da aula anterior);
- $s_i$ é o **peso do exemplo** $i$ (por padrão, $s_i = 1$ para todos) e $S = \sum_i s_i$ (por padrão, $S = n$);
- $C > 0$ é o **inverso da força de regularização**: $C$ pequeno ⇒ penalização forte ⇒ pesos menores. É a mesma ideia do Ridge, com $\lambda$ fazendo o papel de $1/C$;
- o intercepto $b$ não entra na penalização.

*Procedência:* guia do usuário do scikit-learn, seção *Logistic regression*, caso binário (scikit-learn, 2026).

**Como a minimização é feita?** O *solver* padrão é o `lbfgs` (Liu & Nocedal, 1989), um método **quasi-Newton**: ele usa o mesmo gradiente que você implementou no gradiente descendente, mas corrige a direção de descida com uma aproximação da curvatura construída a partir dos últimos gradientes. Por isso converge em dezenas de iterações, em vez de milhares. O processo termina quando o critério de tolerância (`tol`) é satisfeito ou ao atingir `max_iter` iterações.

> **Nota de versão.** A partir do scikit-learn 1.8, o hiperparâmetro `penalty` foi descontinuado em favor de `l1_ratio` (`l1_ratio=0` ⇒ L2). Usaremos apenas o padrão (L2) e o hiperparâmetro `C`, o que funciona em qualquer versão recente.

---
## 8. Divisão treino / validação / teste (estratificada)

Cada subconjunto tem uma função diferente — e misturá-las é a forma mais comum de produzir resultados otimistas:

| Conjunto | Função | Pode influenciar decisões? |
|---|---|---|
| **Treino** (60%) | ajustar os parâmetros (`fit`) | sim |
| **Validação** (20%) | comparar modelos, escolher hiperparâmetros e **limiares** | sim |
| **Teste** (20%) | estimar o desempenho final, **uma única vez** | **não** |

Usamos `stratify` para que cada subconjunto preserve a prevalência $\pi \approx 15{,}5\%$. Sem estratificação, a quantidade de módulos defeituosos em cada parte poderia variar bastante apenas por acaso, o que altera diretamente precisão e recall.

In [ ]:
# 1ª divisão: 80% (treino + validação) e 20% (teste)
X_tv, X_teste, y_tv, y_teste = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)

# 2ª divisão: dos 80%, 25% vão para validação  →  60% / 20% / 20% do total
X_treino, X_val, y_treino, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, stratify=y_tv, random_state=SEED)

divisao = pd.DataFrame({
    "módulos": [len(y_treino), len(y_val), len(y_teste)],
    "com defeito (P)": [y_treino.sum(), y_val.sum(), y_teste.sum()],
    "prevalência": [y_treino.mean(), y_val.mean(), y_teste.mean()],
}, index=["treino", "validação", "teste"])
divisao

Observe que validação e teste têm apenas **65 módulos com defeito** cada. Cada módulo defeituoso encontrado ou perdido altera o recall em $1/65 \approx 1{,}5$ ponto percentual. **Métricas calculadas sobre poucos positivos são ruidosas** — voltaremos a isso na Seção 16.

> **Regra desta aula:** o conjunto de teste (`X_teste`, `y_teste`) só será usado na investigação final.

---
## 9. A linha de base: um classificador que nunca emite alarme

Todo modelo deve ser comparado com uma **linha de base** (*baseline*) que não usa os atributos. O `DummyClassifier(strategy="most_frequent")` aprende, no `fit`, qual é a classe mais frequente em `y_treino` e a prevê para qualquer entrada.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_treino, y_treino)

yhat_val_dummy = dummy.predict(X_val)
print("Classe prevista pelo dummy:", np.unique(yhat_val_dummy))
print(f"Acurácia do dummy na validação: {accuracy_score(y_val, yhat_val_dummy):.3f}")
print(f"1 − prevalência na validação:   {1 - y_val.mean():.3f}")

Os dois números coincidem, e não por acaso. Se $\hat y_i = 0$ para todo $i$, a acurácia vira

$$
\frac{1}{n}\sum_{i=1}^{n} \mathbb{I}[y_i = \hat y_i] = \frac{1}{n}\sum_{i=1}^{n} \mathbb{I}[y_i = 0] = 1 - \pi.
$$

Em qualquer problema com prevalência $\pi$, **um classificador que ignora os dados tem acurácia $\max(\pi, 1 - \pi)$**. Com $\pi = 0{,}155$, isso significa ≈ 84,6% "de graça".

---
## 10. O primeiro classificador real: `StandardScaler` → `LogisticRegression`

In [ ]:
logreg = make_pipeline(
    StandardScaler(),                       # aprende μ_j e s_j no treino
    LogisticRegression(C=1.0, max_iter=1000),
)
logreg.fit(X_treino, y_treino)

scaler, clf = logreg.named_steps["standardscaler"], logreg.named_steps["logisticregression"]
print("classes_:           ", clf.classes_)
print("coef_ (forma):      ", clf.coef_.shape, "  → um peso por atributo")
print("intercept_:         ", clf.intercept_.round(3))
print("iterações do lbfgs: ", clf.n_iter_)

### 10.1 Abrindo a caixa: reproduzindo o `Pipeline` com NumPy

Vamos calcular à mão, para os 5 primeiros módulos da validação, cada etapa da tabela da Seção 7.1 e comparar com os métodos do `Pipeline`.

In [ ]:
X5 = X_val.iloc[:5].to_numpy()                      # (5, 21)

X5_pad = (X5 - scaler.mean_) / scaler.scale_        # padronização com μ e s do TREINO
z = X5_pad @ clf.coef_.ravel() + clf.intercept_[0]  # logits: z = wᵀx̃ + b   → (5,)
p_hat = 1 / (1 + np.exp(-z))                        # sigmoide               → (5,)
y_hat = (z > 0).astype(int)                         # regra do predict       → (5,)

comparacao = pd.DataFrame({
    "z (NumPy)": z,
    "decision_function": logreg.decision_function(X_val.iloc[:5]),
    "p̂ (NumPy)": p_hat,
    "predict_proba[:, 1]": logreg.predict_proba(X_val.iloc[:5])[:, 1],
    "ŷ (NumPy)": y_hat,
    "predict": logreg.predict(X_val.iloc[:5]),
    "y real": y_val.iloc[:5].to_numpy(),
})
display(comparacao)
print("Máxima diferença em p̂:", np.abs(p_hat - logreg.predict_proba(X_val.iloc[:5])[:, 1]).max())

### O que acabamos de observar?

- O `Pipeline` executa **exatamente** a matemática da regressão logística: as diferenças são da ordem do erro de ponto flutuante.
- `predict_proba` devolve uma matriz $n \times 2$; a coluna `[:, 1]` corresponde à classe `classes_[1] = 1` (defeito).
- `predict` aplica o limiar fixo $0{,}5$ sobre $\hat p$. **O limiar não faz parte do modelo ajustado**: é uma decisão que podemos (e frequentemente devemos) mudar. Esse é o tema da Seção 13.

> **Cuidado com a interpretação dos coeficientes.** Muitos atributos são quase redundantes (por exemplo, a correlação entre `v(g)` e `branchCount` é ≈ 0,999). Com multicolinearidade — lembre-se da aula de Ridge —, pesos de atributos correlacionados podem ter sinais opostos e instáveis. Não leia um peso negativo como "complexidade reduz defeitos".

In [ ]:
yhat_val_lr = logreg.predict(X_val)            # classes com limiar 0,5
p_val_lr = logreg.predict_proba(X_val)[:, 1]   # escores: P̂(defeito | x)

print(f"Acurácia na validação — dummy:               {accuracy_score(y_val, yhat_val_dummy):.3f}")
print(f"Acurácia na validação — regressão logística: {accuracy_score(y_val, yhat_val_lr):.3f}")

### O que acabamos de observar?

A regressão logística com 21 atributos tem acurácia praticamente igual à do classificador que **nunca** aponta defeito (diferença de 1 módulo em 422). Isso quer dizer que o modelo é inútil? **A acurácia não consegue responder.** Ela mistura, em um único número, acertos e erros de naturezas muito diferentes. Precisamos separar os tipos de erro.

### Checkpoint 1

Antes de continuar, você deve conseguir responder:

1. Quais parâmetros o `fit` ajusta em cada etapa do `Pipeline`? Quais valores são hiperparâmetros?
2. Qual é a diferença entre `decision_function`, `predict_proba` e `predict`? Qual deles depende de um limiar?
3. Por que o `StandardScaler` precisa ser ajustado apenas com o treino?
4. Por que 84,6% de acurácia do dummy não é um bom resultado neste problema?

---
# Parte III — A matriz de confusão e as métricas pontuais

## 11. A matriz de confusão

No controle de qualidade de software, os dois erros possíveis têm consequências muito diferentes:

- **Falso negativo (FN):** o módulo tem defeito, mas o classificador diz que não. O defeito **escapa** para produção — em um sistema de solo de satélites.
- **Falso positivo (FP):** o módulo não tem defeito, mas o classificador manda inspecioná-lo. Horas de revisão são **desperdiçadas** (um **alarme falso**).

Tomando a classe **1 (defeito)** como **positiva**, definimos as quatro contagens (Fawcett, 2006):

$$
\begin{aligned}
TP &= \sum_{i=1}^{n} \mathbb{I}[y_i = 1 \,\wedge\, \hat y_i = 1], &\qquad
FN &= \sum_{i=1}^{n} \mathbb{I}[y_i = 1 \,\wedge\, \hat y_i = 0], \\
FP &= \sum_{i=1}^{n} \mathbb{I}[y_i = 0 \,\wedge\, \hat y_i = 1], &\qquad
TN &= \sum_{i=1}^{n} \mathbb{I}[y_i = 0 \,\wedge\, \hat y_i = 0],
\end{aligned}
$$

em que $\mathbb{I}[\cdot]$ vale 1 se a condição é verdadeira e 0 caso contrário. Relações úteis: $TP + FN = P$ (positivos reais), $FP + TN = N$ (negativos reais), $TP + FP$ = alarmes emitidos e $TP + FN + FP + TN = n$.

No scikit-learn, **as linhas são as classes reais e as colunas são as classes previstas**, na ordem dos rótulos $[0, 1]$ (scikit-learn, 2026, *Confusion matrix*):

|  | previsto $\hat y = 0$ | previsto $\hat y = 1$ |
|---|---|---|
| **real $y = 0$** | TN | FP |
| **real $y = 1$** | FN | TP |

> **Atenção à convenção.** Fawcett (2006) e o próprio cabeçalho do arquivo KC1 desenham a matriz com as previsões nas linhas. Sempre confira a convenção antes de ler uma matriz de confusão.

### 11.1 Exemplo manual com 10 módulos

| módulo $i$ | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|
| real $y_i$ | 1 | 0 | 0 | 1 | 0 | 0 | 1 | 0 | 0 | 0 |
| previsto $\hat y_i$ | 1 | 0 | 1 | 0 | 0 | 0 | 1 | 0 | 0 | 1 |
| tipo | TP | TN | FP | FN | TN | TN | TP | TN | TN | FP |

Contando: $TP = 2$, $FN = 1$, $FP = 2$, $TN = 5$. Conferência: $P = TP + FN = 3$ e $N = FP + TN = 7$.

Agora a mesma conta em NumPy, sem scikit-learn:

In [ ]:
def matriz_confusao(y_true, y_pred):
    # Matriz de confusão binária [[TN, FP], [FN, TP]] (linhas = real, colunas = previsto).
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    return np.array([[tn, fp], [fn, tp]])

y_toy    = np.array([1, 0, 0, 1, 0, 0, 1, 0, 0, 0])
yhat_toy = np.array([1, 0, 1, 0, 0, 0, 1, 0, 0, 1])

print("NumPy:\n", matriz_confusao(y_toy, yhat_toy))
print("scikit-learn:\n", confusion_matrix(y_toy, yhat_toy))

tn, fp, fn, tp = confusion_matrix(y_toy, yhat_toy).ravel()   # ordem do ravel: TN, FP, FN, TP
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")

### 11.2 Matrizes de confusão no KC1 (validação)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, yhat, nome in [(axes[0], yhat_val_dummy, "Dummy (nunca alarma)"),
                       (axes[1], yhat_val_lr, "Regressão logística (limiar 0,5)")]:
    ConfusionMatrixDisplay.from_predictions(
        y_val, yhat, display_labels=["sem defeito", "com defeito"],
        cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(f"{nome}\nacurácia = {accuracy_score(y_val, yhat):.3f}")
    ax.set_xlabel("classe prevista")
    ax.set_ylabel("classe real")
    ax.grid(False)
plt.tight_layout()
plt.show()

### O que acabamos de observar?

- **Dummy:** $TN = 357$, $FP = 0$, $FN = 65$, $TP = 0$. Nenhum alarme falso — e nenhum defeito encontrado. Os 65 defeitos da validação escapam.
- **Regressão logística:** $TN = 346$, $FP = 11$, $FN = 53$, $TP = 12$. O modelo encontra 12 dos 65 módulos defeituosos, ao custo de 11 alarmes falsos.

A acurácia é $(TP + TN)/n$. Ao passar do dummy para a regressão logística, ganhamos 12 TP e perdemos 11 TN: o numerador muda em apenas uma unidade. **A acurácia trata os dois erros como equivalentes e é dominada pela classe majoritária**, por isso não distingue um detector que encontra defeitos de um que não encontra nenhum.

---
## 12. Métricas derivadas da matriz de confusão

Cada métrica responde a uma **pergunta diferente**. As fórmulas abaixo seguem Fawcett (2006, Fig. 1) e a documentação do scikit-learn (scikit-learn, 2026, *Metrics and scoring*). Na literatura de predição de defeitos, recall e taxa de falsos positivos aparecem com os nomes *pd* (*probability of detection*) e *pf* (*probability of false alarm*) (Menzies, Greenwald & Frank, 2007).

| Métrica | Fórmula | Pergunta que responde (no KC1) | Interpretação probabilística |
|---|---|---|---|
| **Acurácia** | $\dfrac{TP + TN}{n}$ | Que fração das decisões está correta? | $P(\hat y = y)$ |
| **Precisão** | $\dfrac{TP}{TP + FP}$ | Dos módulos que mandamos inspecionar, quantos tinham defeito? | $P(y = 1 \mid \hat y = 1)$ |
| **Recall** (sensibilidade, TPR, *pd*) | $\dfrac{TP}{TP + FN}$ | Dos módulos com defeito, quantos encontramos? | $P(\hat y = 1 \mid y = 1)$ |
| **Especificidade** (TNR) | $\dfrac{TN}{TN + FP}$ | Dos módulos sem defeito, quantos deixamos em paz? | $P(\hat y = 0 \mid y = 0)$ |
| **Taxa de falsos positivos** (FPR, *pf*) | $\dfrac{FP}{FP + TN} = 1 - \text{especificidade}$ | Dos módulos sem defeito, quantos geraram alarme falso? | $P(\hat y = 1 \mid y = 0)$ |

As probabilidades da última coluna são as quantidades que as frações **estimam** na população de onde vieram os dados.

### 12.1 Por que precisão e recall caminham em sentidos opostos?

Precisão e recall têm o **mesmo numerador** (TP) e denominadores diferentes:

- a **precisão** é penalizada pelos **FP** (alarmes falsos);
- o **recall** é penalizado pelos **FN** (defeitos que escapam).

Um detector que alarma em **tudo** tem recall $= 1$ e precisão $= \pi$. Um detector que alarma apenas no módulo mais suspeito pode ter precisão $= 1$ e recall $\approx 0$. Nenhum dos extremos é útil; por isso precisamos de métricas que combinem os dois.

### 12.2 F1 e $F_\beta$

O **F1** é a média harmônica de precisão ($\mathrm{Prec}$) e recall ($\mathrm{Rec}$):

$$
F_1 = \frac{2 \cdot \mathrm{Prec} \cdot \mathrm{Rec}}{\mathrm{Prec} + \mathrm{Rec}} = \frac{2\,TP}{2\,TP + FP + FN}.
$$

Pela Seção 4.3, o F1 só é alto quando **ambos** são altos. Note que **o F1 não usa TN**: a enorme massa de negativos corretos não o infla, o que é bom em problemas desbalanceados, mas também significa que o F1 muda se trocarmos qual classe é a positiva.

Quando os dois erros não têm a mesma importância, usa-se a generalização $F_\beta$ (van Rijsbergen, 1979; scikit-learn, 2026):

$$
F_\beta = (1 + \beta^2)\,\frac{\mathrm{Prec} \cdot \mathrm{Rec}}{\beta^2 \cdot \mathrm{Prec} + \mathrm{Rec}} = \frac{(1 + \beta^2)\,TP}{(1 + \beta^2)\,TP + FP + \beta^2\,FN},
$$

em que $\beta > 0$ expressa quantas vezes o recall é considerado mais importante que a precisão: $\beta = 2$ favorece o recall; $\beta = 0{,}5$ favorece a precisão; $\beta = 1$ recupera o F1.

### 12.3 Acurácia balanceada

$$
\text{acurácia balanceada} = \frac{1}{2}\left(\frac{TP}{TP + FN} + \frac{TN}{TN + FP}\right),
$$

a média entre recall e especificidade (scikit-learn, 2026, `balanced_accuracy_score`). Qualquer classificador que ignora os dados tem acurácia balanceada $0{,}5$.

### 12.4 Calculando à mão no exemplo de 10 módulos

Com $TP = 2$, $FN = 1$, $FP = 2$, $TN = 5$:

- acurácia $= (2 + 5)/10 = 0{,}700$
- precisão $= 2/(2 + 2) = 0{,}500$
- recall $= 2/(2 + 1) \approx 0{,}667$
- especificidade $= 5/(5 + 2) \approx 0{,}714$
- F1 $= 2 \cdot 2/(2 \cdot 2 + 2 + 1) = 4/7 \approx 0{,}571$
- acurácia balanceada $= (0{,}667 + 0{,}714)/2 \approx 0{,}690$

Implementando as quatro primeiras métricas com NumPy, a partir da nossa `matriz_confusao`:

In [ ]:
def acuracia(y_true, y_pred):
    (tn, fp), (fn, tp) = matriz_confusao(y_true, y_pred)
    return (tp + tn) / (tp + tn + fp + fn)

def precisao(y_true, y_pred):
    (tn, fp), (fn, tp) = matriz_confusao(y_true, y_pred)
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0   # sem alarmes: indefinida → 0 (como zero_division=0)

def recall(y_true, y_pred):
    (tn, fp), (fn, tp) = matriz_confusao(y_true, y_pred)
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def f1(y_true, y_pred):
    (tn, fp), (fn, tp) = matriz_confusao(y_true, y_pred)
    return 2 * tp / (2 * tp + fp + fn) if tp + fp + fn > 0 else 0.0

pd.DataFrame({
    "NumPy":        [acuracia(y_toy, yhat_toy), precisao(y_toy, yhat_toy), recall(y_toy, yhat_toy), f1(y_toy, yhat_toy)],
    "scikit-learn": [accuracy_score(y_toy, yhat_toy), precision_score(y_toy, yhat_toy),
                     recall_score(y_toy, yhat_toy), f1_score(y_toy, yhat_toy)],
}, index=["acurácia", "precisão", "recall", "F1"])

Os valores coincidem com o cálculo manual. Uma observação sobre casos indefinidos: se o classificador nunca emite alarme, $TP + FP = 0$ e a precisão é $0/0$. O scikit-learn controla isso com o argumento `zero_division` (o padrão emite um aviso e devolve 0). Para o dummy, usaremos `zero_division=0` explicitamente.

### 12.5 As métricas no KC1

In [ ]:
def tabela_metricas(preds, y_true):
    # preds: dicionário {nome do modelo: vetor de classes previstas}
    linhas = {}
    for nome, yhat in preds.items():
        linhas[nome] = {
            "acurácia": accuracy_score(y_true, yhat),
            "precisão": precision_score(y_true, yhat, zero_division=0),
            "recall": recall_score(y_true, yhat),
            "especificidade": recall_score(y_true, yhat, pos_label=0),
            "F1": f1_score(y_true, yhat, zero_division=0),
            "acur. balanceada": balanced_accuracy_score(y_true, yhat),
            "% sinalizados": np.mean(yhat),
        }
    return pd.DataFrame(linhas).T

tabela_metricas({"Dummy (most_frequent)": yhat_val_dummy,
                 "LogReg (limiar 0,5)": yhat_val_lr}, y_val)

### O que acabamos de observar?

- Agora as diferenças aparecem. O dummy tem recall, precisão e F1 iguais a zero e acurácia balanceada 0,5; a regressão logística tem precisão ≈ 0,52, recall ≈ 0,18 e F1 ≈ 0,27.
- A especificidade da regressão logística é alta (≈ 0,97) porque ela **quase nunca alarma**: sinaliza apenas ≈ 5,5% dos módulos, enquanto 15,4% têm defeito.
- Com limiar 0,5, o modelo é um **detector tímido**: quando alarma, acerta cerca de metade das vezes, mas deixa escapar mais de 80% dos defeitos. Se isso é aceitável depende de quantos módulos a equipe consegue inspecionar — é o assunto da Seção 13.

O `classification_report` apresenta as mesmas métricas **para cada classe**, tratando cada uma, por vez, como positiva:

In [ ]:
print(classification_report(y_val, yhat_val_lr, target_names=["sem defeito (0)", "com defeito (1)"], digits=3))

Como ler o relatório:

- a linha **com defeito (1)** reproduz precisão, recall e F1 da tabela anterior; a linha **sem defeito (0)** mostra as métricas com a classe 0 como positiva (o recall da classe 0 é a especificidade);
- `support` é o número de exemplos reais de cada classe;
- `macro avg` é a média simples entre as classes; `weighted avg` pondera pelo `support`.

> **Armadilha:** em dados desbalanceados, a `weighted avg` é dominada pela classe majoritária e parece boa (≈ 0,8) mesmo com recall de 0,18 na classe que interessa. Em problemas com mais de duas classes, as médias *macro* e *weighted* são a forma padrão de resumir o desempenho — voltaremos a elas nas próximas aulas.

### 12.6 A precisão depende da prevalência

Recall e FPR são condicionados à **classe real**: descrevem o comportamento do detector *dentro* de cada classe. A precisão é condicionada à **previsão** e, por isso, depende da proporção de positivos. Aplicando o Teorema de Bayes (Seção 4.2) às definições de Fawcett (2006), com $\pi = P(y = 1)$:

$$
\mathrm{Prec} = P(y = 1 \mid \hat y = 1)
= \frac{P(\hat y = 1 \mid y = 1)\,P(y = 1)}{P(\hat y = 1 \mid y = 1)\,P(y = 1) + P(\hat y = 1 \mid y = 0)\,P(y = 0)}
= \frac{\mathrm{TPR} \cdot \pi}{\mathrm{TPR} \cdot \pi + \mathrm{FPR} \cdot (1 - \pi)}.
$$

*Procedência:* identidade obtida do Teorema de Bayes (Russell & Norvig, 2022, cap. 12) aplicado às definições de TPR, FPR e precisão de Fawcett (2006); a sensibilidade da precisão à proporção entre classes é discutida em Fawcett (2006) e Saito & Rehmsmeier (2015).

**Consequência:** o **mesmo detector** (mesmos TPR e FPR) terá precisões diferentes em projetos com taxas de defeito diferentes. Vamos verificar a identidade com os números da validação e visualizar o efeito.

In [ ]:
(tn, fp), (fn, tp) = matriz_confusao(y_val, yhat_val_lr)
tpr_lr, fpr_lr, pi_val = tp / (tp + fn), fp / (fp + tn), y_val.mean()

prec_bayes = tpr_lr * pi_val / (tpr_lr * pi_val + fpr_lr * (1 - pi_val))
print(f"TPR = {tpr_lr:.3f}, FPR = {fpr_lr:.3f}, π = {pi_val:.3f}")
print(f"Precisão pela fórmula de Bayes: {prec_bayes:.4f}")
print(f"Precisão medida:                {precision_score(y_val, yhat_val_lr):.4f}")

pis = np.linspace(0.01, 0.60, 200)
prec_curva = tpr_lr * pis / (tpr_lr * pis + fpr_lr * (1 - pis))

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(pis, prec_curva, color=COR_LR, lw=2, label="precisão prevista pela identidade de Bayes")
ax.axhline(tpr_lr, color=COR_BASE, ls=":", lw=1.5, label=f"recall (não depende de π) = {tpr_lr:.2f}")
ax.scatter([pi_val], [prec_bayes], color=COR_ALT, s=60, zorder=3, label=f"KC1 (π = {pi_val:.3f})")
ax.set_xlabel("prevalência de defeitos π")
ax.set_ylabel("valor da métrica")
ax.set_ylim(0, 1)
ax.set_title("Mesmo detector (TPR e FPR fixos), prevalências diferentes")
ax.legend(loc="lower right")
plt.show()

### O que acabamos de observar?

- A identidade reproduz exatamente a precisão medida (ela é uma reescrita das contagens da matriz de confusão).
- A curva é **teórica**: mantém fixos o TPR e o FPR medidos na validação e varia apenas $\pi$. Se o mesmo detector fosse aplicado a um projeto com 5% de módulos defeituosos, sua precisão cairia para cerca de metade; em um projeto com 30%, subiria bastante — **sem nenhuma mudança no modelo**.
- Por isso, **comparar precisões entre conjuntos de dados com prevalências diferentes é enganoso**, enquanto recall e FPR são comparáveis. O cabeçalho do arquivo KC1 registra justamente que a precisão varia muito de um conjunto de dados da NASA para outro (*"Precision's inter-data-set variances vary wildly"*).

---
## Exercício 1 — Da matriz de confusão às métricas

**Objetivo:** consolidar as definições das métricas, o cálculo manual e sua implementação em NumPy.

**Contexto:** uma ferramenta de análise estática avaliou 12 módulos. Os rótulos reais e as previsões estão na célula de código abaixo (`y_ex1` e `yhat_ex1`).

**Tarefa:**

1. **À mão** (na célula de resposta em Markdown): monte a matriz de confusão e calcule acurácia, precisão, recall, especificidade, F1 e $F_2$, **mostrando as contas**.
2. **Em NumPy**: implemente `especificidade(y_true, y_pred)` e `f_beta(y_true, y_pred, beta)` usando a função `matriz_confusao` da Seção 11 — sem usar `sklearn.metrics` dentro das funções.
3. Execute a **célula de verificação**, que compara suas funções com o scikit-learn no exemplo do exercício e na validação do KC1.
4. Sem calcular: para a regressão logística da Seção 12.5 (limiar 0,5), o $F_2$ será **maior ou menor** que o F1? Justifique a partir da fórmula de $F_\beta$ e dos valores de precisão e recall do modelo. Depois confirme com sua função.

**Entregável:** contas em Markdown, as duas funções implementadas, a verificação executada sem erros e a justificativa do item 4.

> **Resposta do estudante — item 1 (cálculos manuais):**

---

## ✅ RESPOSTA — Exercício 1, item 1 (cálculos manuais)

**Classificação exemplo por exemplo.** Com $y$ real e $\hat y$ previsto, classe positiva = 1:

| $i$ | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| $y_i$ | 0 | 1 | 1 | 0 | 0 | 1 | 0 | 0 | 1 | 0 | 0 | 0 |
| $\hat y_i$ | 0 | 1 | 0 | 0 | 1 | 1 | 0 | 1 | 1 | 0 | 0 | 0 |
| tipo | TN | **TP** | **FN** | TN | **FP** | **TP** | TN | **FP** | **TP** | TN | TN | TN |

**Contagens:** $TP = 3$ (exemplos 2, 6, 9), $FN = 1$ (exemplo 3), $FP = 2$ (exemplos 5, 8), $TN = 6$ (exemplos 1, 4, 7, 10, 11, 12).

**Conferência:** $P = TP + FN = 3 + 1 = 4$ (há quatro 1's em $y$ ✓), $N = FP + TN = 2 + 6 = 8$, e $TP+FN+FP+TN = 12 = n$ ✓.

**Matriz de confusão** (linhas = real, colunas = previsto, convenção do scikit-learn):

$$
\begin{array}{c|cc}
 & \hat y = 0 & \hat y = 1 \\ \hline
y = 0 & TN = 6 & FP = 2 \\
y = 1 & FN = 1 & TP = 3
\end{array}
$$

**As seis métricas, com as contas:**

$$\text{acurácia} = \frac{TP+TN}{n} = \frac{3+6}{12} = \frac{9}{12} = \mathbf{0{,}7500}$$

$$\text{precisão} = \frac{TP}{TP+FP} = \frac{3}{3+2} = \frac{3}{5} = \mathbf{0{,}6000}$$

$$\text{recall} = \frac{TP}{TP+FN} = \frac{3}{3+1} = \frac{3}{4} = \mathbf{0{,}7500}$$

$$\text{especificidade} = \frac{TN}{TN+FP} = \frac{6}{6+2} = \frac{6}{8} = \mathbf{0{,}7500}$$

$$F_1 = \frac{2\,TP}{2\,TP+FP+FN} = \frac{2(3)}{2(3)+2+1} = \frac{6}{9} = \mathbf{0{,}6667}$$

$$F_2 = \frac{(1+\beta^2)\,TP}{(1+\beta^2)\,TP + FP + \beta^2 FN}\bigg|_{\beta=2} = \frac{5(3)}{5(3)+2+4(1)} = \frac{15}{21} = \mathbf{0{,}7143}$$

Conferindo o $F_2$ pela outra forma, com $\mathrm{Prec}=0{,}6$ e $\mathrm{Rec}=0{,}75$:

$$F_2 = (1+4)\frac{(0{,}6)(0{,}75)}{4(0{,}6)+0{,}75} = \frac{5 \times 0{,}45}{2{,}4+0{,}75} = \frac{2{,}25}{3{,}15} = 0{,}7143 \;\checkmark$$

**Leitura do resultado.** Aqui $\mathrm{Rec} > \mathrm{Prec}$ (0,75 contra 0,60): o detector encontra a maioria dos defeitos, mas dois dos cinco alarmes são falsos. Como $\beta = 2$ dá mais peso ao recall — o lado **forte** deste detector —, temos $F_2 = 0{,}714 > F_1 = 0{,}667$. Repare que $F_1$ e $F_2$ ficam **entre** precisão e recall, e sempre mais próximos do menor dos dois do que a média aritmética ($0{,}675$) ficaria: é o efeito da média harmônica.

---

In [ ]:
# Exercício 1 — dados do exemplo (não altere)
y_ex1    = np.array([0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0])
yhat_ex1 = np.array([0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0])

# ✅ RESPOSTA — Exercício 1, item 2
def especificidade(y_true, y_pred):
    """TN / (TN + FP) = P(ŷ=0 | y=0). Devolve 0,0 se não houver negativos reais."""
    (tn, fp), (fn, tp) = matriz_confusao(y_true, y_pred)
    return tn / (tn + fp) if (tn + fp) > 0 else 0.0

def f_beta(y_true, y_pred, beta):
    """F_beta na forma com contagens (Seção 12.2):

        F_beta = (1 + beta²)·TP / [ (1 + beta²)·TP + FP + beta²·FN ]

    Equivalente à forma com precisão e recall, mas sem divisões intermediárias —
    o que evita 0/0 quando o detector não emite alarme algum.
    """
    (tn, fp), (fn, tp) = matriz_confusao(y_true, y_pred)
    b2 = beta ** 2
    denominador = (1 + b2) * tp + fp + b2 * fn
    return (1 + b2) * tp / denominador if denominador > 0 else 0.0


# Conferência das contas manuais do item 1
(tn, fp), (fn, tp) = matriz_confusao(y_ex1, yhat_ex1)
print(f"exemplo do exercício: TN={tn} FP={fp} FN={fn} TP={tp}  (manual: 6, 2, 1, 3)")
print(f"  acurácia       = {(tp+tn)/len(y_ex1):.4f}   (manual: 9/12  = 0,7500)")
print(f"  precisão       = {tp/(tp+fp):.4f}   (manual: 3/5   = 0,6000)")
print(f"  recall         = {tp/(tp+fn):.4f}   (manual: 3/4   = 0,7500)")
print(f"  especificidade = {especificidade(y_ex1, yhat_ex1):.4f}   (manual: 6/8   = 0,7500)")
print(f"  F1             = {f_beta(y_ex1, yhat_ex1, 1):.4f}   (manual: 6/9   = 0,6667)")
print(f"  F2             = {f_beta(y_ex1, yhat_ex1, 2):.4f}   (manual: 15/21 = 0,7143)")

# ✅ RESPOSTA — item 4: F1 e F2 da regressão logística na validação
print("\nRegressão logística na validação (limiar 0,5):")
print(f"  precisão = {precision_score(y_val, yhat_val_lr):.4f}")
print(f"  recall   = {recall_score(y_val, yhat_val_lr):.4f}   <- o lado FRACO deste detector")
for beta in [0.5, 1, 2]:
    print(f"  F{beta:<4} = {f_beta(y_val.to_numpy(), yhat_val_lr, beta):.4f}")
print("\nF2 < F1 < F0.5: com recall << precisão, dar mais peso ao recall PIORA o índice.")

In [ ]:
# ✅ Verificação do Exercício 1 — execute sem alterar
try:
    casos = {"exemplo do exercício": (y_ex1, yhat_ex1), "validação KC1 (LogReg)": (y_val.to_numpy(), yhat_val_lr)}
    for nome, (yt, yp) in casos.items():
        ok_esp = np.isclose(especificidade(yt, yp), recall_score(yt, yp, pos_label=0))
        ok_f1 = np.isclose(f_beta(yt, yp, 1), f1_score(yt, yp))
        ok_f2 = np.isclose(f_beta(yt, yp, 2), fbeta_score(yt, yp, beta=2))
        ok_f05 = np.isclose(f_beta(yt, yp, 0.5), fbeta_score(yt, yp, beta=0.5))
        status = "OK" if (ok_esp and ok_f1 and ok_f2 and ok_f05) else "DIFERENTE — revise"
        print(f"{nome:26s} especificidade: {ok_esp} | F1: {ok_f1} | F2: {ok_f2} | F0.5: {ok_f05} → {status}")
except NotImplementedError as e:
    print("⚠️ Ainda não implementado:", e)

> **Resposta do estudante — item 4:**

---

## ✅ RESPOSTA — Exercício 1, item 4

**Previsão antes de calcular: $F_2$ será MENOR que $F_1$.**

**Raciocínio a partir da fórmula.** $F_\beta$ é uma média harmônica **ponderada**, que puxa o resultado para o lado que recebe mais peso:

$$F_\beta = (1+\beta^2)\,\frac{\mathrm{Prec}\cdot\mathrm{Rec}}{\beta^2\,\mathrm{Prec}+\mathrm{Rec}}.$$

O parâmetro $\beta$ diz quantas vezes o **recall** importa mais que a precisão. Logo:

- se $\mathrm{Rec} > \mathrm{Prec}$, aumentar $\beta$ **sobe** o índice (é o caso do exemplo de 12 módulos: $F_2 = 0{,}714 > F_1 = 0{,}667$);
- se $\mathrm{Rec} < \mathrm{Prec}$, aumentar $\beta$ **desce** o índice.

Na regressão logística da Seção 12.5, com limiar 0,5, temos $\mathrm{Prec} = 0{,}5217$ e $\mathrm{Rec} = 0{,}1846$. O recall é **quase três vezes menor** que a precisão — este é um detector tímido, que acerta razoavelmente quando alarma mas deixa escapar 81% dos defeitos. Como $\beta = 2$ pesa mais justamente a dimensão em que o modelo é pior, $F_2$ tem de ficar **abaixo** de $F_1$.

**Confirmação numérica** (célula acima):

| índice | valor | peso relativo |
|:--|--:|:--|
| $F_{0,5}$ | **0,3822** | favorece a precisão (o lado forte) |
| $F_1$ | 0,2727 | pesos iguais |
| $F_2$ | **0,2120** | favorece o recall (o lado fraco) |

A previsão se confirma: $F_2 = 0{,}2120 < F_1 = 0{,}2727$. E a ordenação completa $F_{0,5} > F_1 > F_2$ é consequência direta de $\mathrm{Prec} > \mathrm{Rec}$ — o $F_\beta$ é monotônico em $\beta$, decrescente sempre que a precisão supera o recall.

**Consequência prática que isso revela.** Escolher $\beta$ **não** é um detalhe estético: é declarar qual erro custa mais. Em predição de defeitos, deixar um defeito escapar (FN) custa muito mais que uma revisão desnecessária (FP), então o índice apropriado é $F_2$ ou $F_3$ — e é exatamente por isso que esse modelo, com limiar 0,5, tem uma nota ruim na métrica certa (0,2120) e uma nota menos ruim na métrica errada (0,3822). Otimizar $F_1$ aqui esconderia o problema. A solução não é trocar a métrica: é **mover o limiar** (Seção 13), o que o Exercício 3 faz.

---

---
## Exercício 2 — Detectores ingênuos e a regra de McCabe

**Objetivo:** relacionar as fórmulas das métricas com o comportamento de detectores degenerados e de um sistema baseado em regras, e identificar qual métrica expõe a falha de cada um.

**Contexto:** além do dummy "nunca alarma" e da regressão logística, considere dois detectores:

- **(A) "Inspecionar tudo":** `DummyClassifier(strategy="constant", constant=1)`, que prevê defeito para todo módulo;
- **(B) Regra de McCabe:** sinalizar o módulo se `v(g) > 10`. McCabe (1976) sugeriu 10 como limite superior razoável para a complexidade ciclomática de um módulo, e a documentação do KC1 cita essa regra como padrão para identificar módulos propensos a falhas.

**Tarefa:**

1. **Antes de executar qualquer código**, escreva em função de $\pi$ (prevalência da validação) as expressões de acurácia, precisão, recall, especificidade e F1 do detector A. Mostre o raciocínio a partir das contagens TP, FP, FN, TN.
2. Implemente os detectores A (ajustado com o treino) e B (aplicado a `X_val["v(g)"]`) e monte, com a função `tabela_metricas` da Seção 12.5, uma tabela comparando **quatro** detectores na validação: dummy "nunca alarma", A, B e regressão logística (limiar 0,5).
3. Compare os valores do detector A com suas expressões do item 1.
4. Responda: (a) para cada um dos quatro detectores, qual métrica, **sozinha**, faria o detector parecer bom, e qual métrica expõe sua falha? (b) A regra de McCabe é melhor ou pior que a regressão logística com limiar 0,5? Justifique com pelo menos duas métricas e com a fração de módulos sinalizados.

**Entregável:** expressões do item 1, tabela com os quatro detectores e respostas do item 4.

> **Resposta do estudante — item 1 (expressões em função de π):**

---

## ✅ RESPOSTA — Exercício 2, item 1 (expressões em função de $\pi$)

**Escritas antes de rodar qualquer código.** O detector A sinaliza **todo** módulo: $\hat y_i = 1$ para todo $i$. As quatro contagens saem direto das definições:

- $TP = P$ — todo positivo real é sinalizado, então todos são verdadeiros positivos;
- $FN = 0$ — nada é previsto como 0, então não existe negativo previsto errado;
- $FP = N$ — todo negativo real é sinalizado, então todos são alarmes falsos;
- $TN = 0$ — pelo mesmo motivo de $FN$.

Com $n = P + N$ e $\pi = P/n$ (logo $P = \pi n$ e $N = (1-\pi)n$):

$$\text{acurácia} = \frac{TP+TN}{n} = \frac{P + 0}{n} = \frac{P}{n} = \boxed{\pi}$$

$$\text{precisão} = \frac{TP}{TP+FP} = \frac{P}{P+N} = \frac{P}{n} = \boxed{\pi}$$

$$\text{recall} = \frac{TP}{TP+FN} = \frac{P}{P+0} = \boxed{1}$$

$$\text{especificidade} = \frac{TN}{TN+FP} = \frac{0}{0+N} = \boxed{0}$$

$$F_1 = \frac{2\,TP}{2\,TP+FP+FN} = \frac{2P}{2P+N} = \frac{2\pi n}{2\pi n + (1-\pi)n} = \boxed{\frac{2\pi}{1+\pi}}$$

E, de bônus, a acurácia balanceada: $\tfrac12(\text{recall} + \text{especificidade}) = \tfrac12(1+0) = \boxed{0{,}5}$ — o mesmo valor do dummy que nunca alarma, o que já é um aviso de que **os dois extremos são igualmente inúteis**.

**Dois fatos que essas expressões tornam evidentes:**

1. **Precisão $=\pi$ não é coincidência.** Sinalizar tudo é o mesmo que escolher um módulo ao acaso da população: a chance de ele ter defeito é a prevalência. É exatamente por isso que a linha de base da curva precisão–recall (Seção 14.7) é $\pi$, e não 0,5;
2. **a acurácia do detector A é $\pi$ e a do dummy é $1-\pi$.** Em um problema desbalanceado como o KC1 ($\pi \approx 0{,}154$), os dois detectores degenerados ficam nos extremos opostos da acurácia (0,154 e 0,846) sem que nenhum deles use **um único atributo**. Esse par de números é a demonstração mais curta de que a acurácia, sozinha, não mede capacidade de detecção.

Substituindo $\pi = 0{,}1540$ (prevalência da validação), as previsões numéricas são: acurácia $= 0{,}1540$, precisão $= 0{,}1540$, recall $= 1{,}0000$, especificidade $= 0{,}0000$, $F_1 = 2(0{,}1540)/1{,}1540 = 0{,}2669$.

---

In [ ]:
# ✅ RESPOSTA — Exercício 2, item 2
# Detector A: "inspecionar tudo" — prevê a constante 1 para qualquer entrada.
detector_tudo = DummyClassifier(strategy="constant", constant=1)
detector_tudo.fit(X_treino, y_treino)
yhat_val_tudo = detector_tudo.predict(X_val)

# Detector B: regra de McCabe — sinaliza se a complexidade ciclomática passa de 10.
LIMITE_MCCABE = 10
yhat_val_mccabe = (X_val["v(g)"] > LIMITE_MCCABE).astype(int).to_numpy()

tabela_quatro = tabela_metricas({
    "Dummy (nunca alarma)": yhat_val_dummy,
    "A: inspecionar tudo": yhat_val_tudo,
    "B: McCabe v(g) > 10": yhat_val_mccabe,
    "LogReg (limiar 0,5)": yhat_val_lr,
}, y_val)
display(tabela_quatro)

# ✅ RESPOSTA — item 3: conferência das expressões do item 1 para o detector A
pi_val_ex2 = y_val.mean()
previsto = {"acurácia": pi_val_ex2, "precisão": pi_val_ex2, "recall": 1.0,
            "especificidade": 0.0, "F1": 2 * pi_val_ex2 / (1 + pi_val_ex2),
            "acur. balanceada": 0.5, "% sinalizados": 1.0}
conferencia = pd.DataFrame({
    "previsto (item 1)": previsto,
    "medido": tabela_quatro.loc["A: inspecionar tudo"],
})
conferencia["bate?"] = np.isclose(conferencia["previsto (item 1)"], conferencia["medido"])
print(f"π (validação) = {pi_val_ex2:.4f}")
display(conferencia)
assert conferencia["bate?"].all(), "alguma expressão do item 1 não confere"
print("Todas as expressões do item 1 conferem. ✔")

# Contagens dos dois detectores novos, para a discussão do item 4
for nome, yhat in [("A: inspecionar tudo", yhat_val_tudo), ("B: McCabe v(g) > 10", yhat_val_mccabe)]:
    (tn, fp), (fn, tp) = matriz_confusao(y_val, yhat)
    print(f"{nome:22s} TN={tn:3d} FP={fp:3d} FN={fn:3d} TP={tp:3d}")
print(f"\nMódulos com v(g) > 10 na validação: {yhat_val_mccabe.sum()} de {len(y_val)} "
      f"({yhat_val_mccabe.mean():.1%})")
print(f"AUC de v(g) usada como escore contínuo: {roc_auc_score(y_val, X_val['v(g)']):.4f}")

> **Resposta do estudante — itens 3 e 4:**

---

## ✅ RESPOSTA — Exercício 2, itens 3 e 4

### Item 3 — as expressões conferem

| métrica | previsto no item 1 | medido | confere |
|:--|--:|--:|:--:|
| acurácia | $\pi = 0{,}1540$ | 0,1540 | ✔ |
| precisão | $\pi = 0{,}1540$ | 0,1540 | ✔ |
| recall | $1$ | 1,0000 | ✔ |
| especificidade | $0$ | 0,0000 | ✔ |
| $F_1$ | $2\pi/(1+\pi) = 0{,}2669$ | 0,2669 | ✔ |
| acur. balanceada | $0{,}5$ | 0,5000 | ✔ |

Nenhuma surpresa — e é isso que importa: as métricas do detector A são **determinadas pela prevalência**, não pelos dados. Nenhum atributo foi consultado.

### A tabela dos quatro detectores (validação, 422 módulos, 65 com defeito)

| detector | acurácia | precisão | recall | especif. | $F_1$ | ac. bal. | % sinalizados | TP | FP | FN |
|:--|--:|--:|--:|--:|--:|--:|--:|--:|--:|--:|
| Dummy (nunca alarma) | **0,8460** | 0,0000 | 0,0000 | **1,0000** | 0,0000 | 0,5000 | 0,0% | 0 | 0 | 65 |
| A: inspecionar tudo | 0,1540 | 0,1540 | **1,0000** | 0,0000 | 0,2669 | 0,5000 | 100,0% | 65 | 357 | 0 |
| B: McCabe $v(g)>10$ | 0,8318 | 0,4000 | 0,1846 | 0,9496 | 0,2526 | 0,5671 | 7,1% | 12 | 18 | 53 |
| LogReg (limiar 0,5) | 0,8483 | **0,5217** | 0,1846 | 0,9692 | **0,2727** | **0,5769** | 5,5% | 12 | 11 | 53 |

### Item 4(a) — a métrica que engana e a métrica que denuncia

| detector | parece bom por… | é denunciado por… |
|:--|:--|:--|
| **Dummy (nunca alarma)** | **acurácia 0,8460** e **especificidade 1,0000** — "acerta 85% e nunca incomoda ninguém" | **recall 0,0000** e $F_1 = 0$: não encontra **nenhum** dos 65 defeitos. A acurácia balanceada (0,5) também expõe |
| **A: inspecionar tudo** | **recall 1,0000** — "não deixa passar nada", o sonho do gerente de qualidade | **precisão 0,1540** e **especificidade 0,0000**: 357 alarmes falsos, 100% da frota inspecionada. Acurácia 0,154, a pior das quatro |
| **B: McCabe $v(g)>10$** | **especificidade 0,9496** e acurácia 0,8318 — parecem respeitáveis | **recall 0,1846**: encontra 12 de 65. E a precisão 0,4000 mostra que 18 dos 30 alarmes são falsos |
| **LogReg (limiar 0,5)** | **acurácia 0,8483** (a maior da tabela) e **precisão 0,5217** | **recall 0,1846**: deixa escapar 53 dos 65 defeitos. A acurácia só é alta porque ela herda os 0,846 "de graça" do desbalanceamento |

O padrão é o ponto do exercício: **toda métrica isolada tem um detector degenerado que a maximiza**. Especificidade é maximizada por quem nunca alarma; recall, por quem alarma sempre; acurácia, pela classe majoritária. É por isso que a avaliação precisa de **ao menos um par** de métricas com erros opostos (precisão e recall, ou recall e FPR) mais a fração sinalizada, que diz quanto trabalho a política gera.

### Item 4(b) — McCabe ou regressão logística?

**A regressão logística é melhor, mas a vantagem é mais modesta do que parece.**

As duas têm **exatamente o mesmo recall (0,1846)**: ambas encontram **12 dos 65** módulos defeituosos. A diferença está no custo de chegar lá:

1. **precisão: 0,5217 contra 0,4000.** A LogReg emite 23 alarmes e acerta 12; McCabe emite 30 e acerta os mesmos 12. São **7 revisões desnecessárias a mais** (18 FP contra 11 FP) para o mesmo resultado;
2. **fração sinalizada: 5,5% contra 7,1%.** A LogReg entrega o mesmo recall consumindo **menos capacidade** da equipe — o que, em um problema com orçamento de inspeção, é exatamente a moeda que importa;
3. **acurácia balanceada: 0,5769 contra 0,5671** — mesma conclusão, agora resumida em um número que não é inflado pela classe majoritária.

**Mas a comparação, como posta, é injusta com a regra de McCabe, e vale registrar por quê.** O limiar 10 é um valor fixo da literatura de 1976, enquanto o limiar 0,5 da LogReg é igualmente arbitrário — nenhum dos dois foi escolhido para este problema. Se eu tratar $v(g)$ como um **escore** e varrer seus limiares (como no Exercício 3), a AUC de $v(g)$ sozinha é **0,7195**, contra 0,8034 da LogReg: a vantagem real do modelo está na **ordenação**, não neste ponto de operação específico. E há uma ressalva mais incômoda ainda, que o Exercício 4 vai medir: ordenar por `loc` (uma única coluna, linhas de código) alcança AUC 0,8018 — praticamente a mesma da regressão logística com 21 atributos.

**Veredito prático:** entre os quatro, eu ficaria com a LogReg, mas **não** no limiar 0,5 — nenhum dos quatro detectores desta tabela é utilizável, porque todos ou deixam escapar >80% dos defeitos ou inspecionam a frota inteira. O que falta não é um modelo melhor: é escolher o **ponto de operação** a partir da capacidade da equipe. É o assunto do Exercício 3.

---

### checkpoint 2

1. Qual métrica responde "dos alarmes emitidos, quantos eram verdadeiros"? E "dos defeitos existentes, quantos encontramos"?
2. Por que o F1 não usa TN? Em que situação isso é uma vantagem?
3. Se a prevalência de defeitos cair pela metade e o detector continuar o mesmo, o que acontece com recall, FPR e precisão?
4. Por que a `weighted avg` do `classification_report` pode enganar em dados desbalanceados?

---
# Parte IV — O limiar de decisão

## 13. Escolhendo o ponto de operação

Até aqui usamos a regra do `predict`, com limiar fixo em 0,5. De forma geral, para um limiar $t \in [0, 1]$:

$$
\hat y(t) = \mathbb{I}\big[\hat p(\mathbf{x}) \ge t\big].
$$

**Por que 0,5?** Se $\hat p(\mathbf{x})$ fosse a probabilidade verdadeira de defeito e os dois erros custassem o mesmo, sinalizar quando $\hat p \ge 0{,}5$ minimizaria o número esperado de erros (Elkan, 2001). Nenhuma das duas condições precisa valer no nosso problema: um defeito que escapa costuma custar muito mais que uma revisão desnecessária.

**O que acontece quando $t$ diminui?** O conjunto de módulos sinalizados $\{i : \hat p_i \ge t\}$ só pode **crescer**. Logo:

- TP e FP **nunca diminuem**; FN e TN **nunca aumentam**;
- o **recall** $TP/P$ e a **FPR** $FP/N$ **nunca diminuem** (os denominadores $P$ e $N$ são fixos);
- a **precisão** $TP/(TP+FP)$ **não tem comportamento monótono garantido**: numerador e denominador mudam juntos.

> **Pare e pense (não precisa entregar):** o que você espera que aconteça com TP, FP, recall, precisão e fração de módulos sinalizados quando o limiar passar de 0,5 para 0,3, 0,2 e 0,1? Registre sua expectativa mentalmente antes de executar as próximas células.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
bins = np.linspace(0, 1, 41)
# density=True normaliza cada classe separadamente: compara formas, apesar de 357 vs. 65 módulos
ax.hist(p_val_lr[y_val == 0], bins=bins, alpha=0.6, color=COR_LR, density=True, label="sem defeito (y = 0)")
ax.hist(p_val_lr[y_val == 1], bins=bins, alpha=0.6, color=COR_ALT, density=True, label="com defeito (y = 1)")
for t, estilo in [(0.5, "-"), (0.2, "--")]:
    ax.axvline(t, color="black", ls=estilo, lw=1.2, label=f"limiar t = {t}")
ax.set_xlabel("probabilidade estimada de defeito  p̂(x)")
ax.set_ylabel("densidade (por classe)")
ax.set_title("Distribuição dos escores da regressão logística na validação")
ax.legend()
plt.show()

frac_abaixo = np.mean(p_val_lr[y_val == 1] < 0.5)
print(f"Fração dos módulos defeituosos com p̂ < 0,5: {frac_abaixo:.1%}")
p_treino_lr = logreg.predict_proba(X_treino)[:, 1]
print(f"Média de p̂ no treino: {p_treino_lr.mean():.4f}  |  prevalência no treino: {y_treino.mean():.4f}")

### O que acabamos de observar?

- Cerca de **80% dos módulos defeituosos têm $\hat p < 0{,}5$**. Com limiar 0,5, eles não são sinalizados — daí o recall baixo.
- Isso não é um defeito de implementação. A média de $\hat p$ no treino é **igual** à prevalência do treino. A razão está na otimização: como o intercepto $b$ não é penalizado, no ótimo a derivada do objetivo em relação a $b$ é zero, e para a entropia cruzada essa derivada é $\frac{1}{n}\sum_i (\hat p_i - y_i)$ (o mesmo gradiente da aula anterior). Logo $\frac{1}{n}\sum_i \hat p_i = \frac{1}{n}\sum_i y_i = \pi$. Com $\pi \approx 0{,}155$, as probabilidades estimadas ficam, em geral, baixas.
- As duas distribuições **se sobrepõem**: qualquer limiar que capture mais defeitos também captura mais módulos sem defeito.

Vamos quantificar o compromisso para alguns limiares:

In [ ]:
def metricas_no_limiar(y_true, escores, t):
    yhat = (escores >= t).astype(int)
    (tn, fp), (fn, tp) = matriz_confusao(y_true, yhat)
    return {"limiar": t, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "% sinalizados": yhat.mean(),
            "precisão": precision_score(y_true, yhat, zero_division=0),
            "recall": recall_score(y_true, yhat),
            "F1": f1_score(y_true, yhat, zero_division=0)}

pd.DataFrame([metricas_no_limiar(y_val, p_val_lr, t) for t in [0.5, 0.4, 0.3, 0.2, 0.1]]).set_index("limiar")

### O que acabamos de observar?

- De $t = 0{,}5$ para $t = 0{,}2$: o recall sobe de ≈ 0,18 para ≈ 0,62, a precisão cai de ≈ 0,52 para ≈ 0,37 e a fração de módulos sinalizados vai de ≈ 5,5% para ≈ 26%.
- TP e FP crescem juntos, como previsto. A precisão, neste caso, oscila levemente antes de cair — sem garantia de monotonicidade.
- O F1 é maior em limiares intermediários. Mas **maximizar o F1 não é, por si só, um objetivo de engenharia**: o F1 assume uma importância relativa fixa entre precisão e recall, que pode não corresponder aos custos reais nem à capacidade da equipe.

**O mesmo modelo, com os mesmos parâmetros $\mathbf{w}$ e $b$, gera detectores muito diferentes dependendo de $t$.** A escolha do limiar deve vir do problema. Veremos duas formas de fazer isso.

### 13.1 Limiar a partir de custos

Suponha que um alarme falso custe $c_{FP}$ (uma revisão desnecessária) e que um defeito que escapa custe $c_{FN}$ (correção em produção), com custo zero para as decisões corretas. Para um módulo com probabilidade de defeito $p$:

- custo esperado de **sinalizar**: $(1 - p)\,c_{FP}$;
- custo esperado de **não sinalizar**: $p\,c_{FN}$.

Sinalizar vale a pena quando $p\,c_{FN} \ge (1 - p)\,c_{FP}$, isto é, quando

$$
p \;\ge\; t^{*} = \frac{c_{FP}}{c_{FP} + c_{FN}}.
$$

*Procedência:* Elkan (2001, Eq. 2), no caso em que decisões corretas têm custo zero. **Hipótese importante:** $\hat p(\mathbf{x})$ precisa ser uma boa estimativa da probabilidade real (modelo **calibrado**). Com $c_{FN} = 10\,c_{FP}$, temos $t^{*} = 1/11 \approx 0{,}091$.

In [ ]:
c_FP, c_FN = 1.0, 10.0              # custos relativos: 1 revisão desnecessária vs. 1 defeito que escapa
t_custo = c_FP / (c_FP + c_FN)

linhas = []
for t in [0.5, 0.3, 0.2, 0.1, t_custo]:
    m = metricas_no_limiar(y_val, p_val_lr, t)
    m["custo total"] = c_FP * m["FP"] + c_FN * m["FN"]
    linhas.append(m)
tabela_custos = pd.DataFrame(linhas).set_index("limiar")[["FP", "FN", "% sinalizados", "precisão", "recall", "custo total"]]
print(f"t* = {t_custo:.4f}")
tabela_custos

### O que acabamos de observar?

Com $c_{FN} = 10\,c_{FP}$, o limiar $t^{*}$ produz o menor custo total entre os limiares da tabela, mais que duas vezes menor que o custo com $t = 0{,}5$. O preço é sinalizar quase metade dos módulos, com precisão baixa — o que faz sentido quando deixar um defeito escapar é muito caro.

Dois cuidados:

1. A regra funciona bem aqui porque a regressão logística tende a produzir probabilidades razoavelmente calibradas. Veremos na Seção 15 um caso em que $\hat p$ deixa de ser uma probabilidade confiável.
2. O limiar foi escolhido observando a **validação**. Se tivéssemos escolhido observando o teste, a estimativa final de custo seria otimista.

### 13.2 Limiar a partir de um orçamento de inspeção

Muitas vezes a restrição real é de **capacidade**: "a equipe consegue inspecionar no máximo 25% dos módulos". Nesse caso, o limiar é o menor $t$ tal que a fração sinalizada não ultrapasse o orçamento:

$$
t_{\text{orç}} = \min\Big\{\, t \;:\; \frac{1}{n}\sum_{i=1}^{n} \mathbb{I}[\hat p_i \ge t] \le \text{orçamento} \Big\}.
$$

Quando há **empates** nos escores (comum com atributos inteiros como `loc`), um quantil simples pode ultrapassar o orçamento; por isso buscamos entre os valores distintos dos escores.

In [ ]:
def limiar_por_orcamento(escores, orcamento):
    # Menor limiar t (entre os escores distintos) cuja fração sinalizada (escores >= t) não excede o orçamento.
    escores = np.asarray(escores)
    candidatos = np.unique(escores)                               # ordem crescente
    fracao = np.array([np.mean(escores >= c) for c in candidatos])
    return candidatos[fracao <= orcamento].min()

t_orc = limiar_por_orcamento(p_val_lr, 0.25)
m = metricas_no_limiar(y_val, p_val_lr, t_orc)
print(f"Limiar para orçamento de 25%: t = {t_orc:.3f}")
print(f"Sinalizados: {m['% sinalizados']:.1%} | recall = {m['recall']:.3f} | precisão = {m['precisão']:.3f}")

Com 25% dos módulos inspecionados, a regressão logística encontra cerca de 60% dos defeitos da validação. Compare com uma inspeção **aleatória** de 25% dos módulos, que encontraria, em média, 25% dos defeitos: o modelo concentra a inspeção nos módulos mais suspeitos.

> **No scikit-learn:** a partir da versão 1.5 existem `FixedThresholdClassifier`, que embute um limiar escolhido por você no estimador, e `TunedThresholdClassifierCV`, que escolhe o limiar por validação cruzada otimizando uma métrica (scikit-learn, 2026, *Tuning the decision threshold for class prediction*). Nesta aula fazemos a escolha manualmente para entender o que essas classes automatizam.

---
## Exercício 3 — Varredura de limiares com restrição operacional

**Objetivo:** implementar a varredura completa de limiares e escolher um ponto de operação a partir de uma restrição de negócio, usando apenas a validação.

**Contexto:** a gerência de qualidade exige que o detector encontre **pelo menos 70% dos módulos defeituosos** (recall ≥ 0,70). Entre os limiares que satisfazem essa exigência, deseja-se o de **maior precisão** (menos horas desperdiçadas).

**Tarefa:**

1. Implemente `varrer_limiares(y_true, escores)` **em NumPy** (sem `sklearn.metrics`), que avalia **todos** os limiares $t$ iguais aos valores distintos de `escores` e devolve um `DataFrame` com as colunas `limiar`, `TP`, `FP`, `FN`, `TN`, `precisao`, `recall`, `f1` e `frac_sinalizada`, ordenado por limiar crescente.
2. Execute a **célula de verificação** (compara seu resultado com `precision_recall_curve` do scikit-learn).
3. Faça **um** gráfico com precisão, recall e F1 em função do limiar (eixo $y$ em $[0, 1]$), marcando $t = 0{,}5$ e os dois limiares do item 4.
4. Encontre: (a) $t_{F1}$, o limiar que maximiza o F1; (b) $t_{70}$, o limiar de maior precisão com recall ≥ 0,70. Para cada um, informe a matriz de confusão, a precisão, o recall e a fração de módulos sinalizados.
5. Em 4 a 6 frases: qual limiar você recomendaria à gerência e por quê? Por que $t_{F1}$ pode não ser adequado aqui? Por que essa escolha deve ser feita na validação e não no teste?

**Pontos de verificação:** o recall da sua tabela deve ser **não crescente** em função do limiar (confira com `np.all(np.diff(...) <= 0)`); com o menor limiar, `frac_sinalizada` deve ser 1.

**Entregável:** função implementada, verificação executada, gráfico, tabela com $t_{F1}$ e $t_{70}$ e a justificativa.

In [ ]:
# Exercício 3 — itens 1, 3 e 4

# ✅ RESPOSTA — item 1: varredura completa, só com NumPy (sem sklearn.metrics)
def varrer_limiares(y_true, escores):
    """Avalia TODOS os limiares iguais aos valores distintos de `escores`.

    Regra de decisão: ŷ = 1 se escore >= t (a mesma de metricas_no_limiar).
    Devolve um DataFrame ordenado por limiar crescente.
    """
    y_true = np.asarray(y_true)
    escores = np.asarray(escores)
    P = int((y_true == 1).sum())
    N = int((y_true == 0).sum())

    linhas = []
    for t in np.unique(escores):                 # np.unique já devolve ordenado
        sinalizado = escores >= t
        tp = int(np.sum(sinalizado & (y_true == 1)))
        fp = int(np.sum(sinalizado & (y_true == 0)))
        fn = P - tp                              # P e N são fixos: o resto sai por subtração
        tn = N - fp
        precisao = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall_t = tp / P if P > 0 else 0.0
        f1_t = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0
        linhas.append((t, tp, fp, fn, tn, precisao, recall_t, f1_t, sinalizado.mean()))

    return pd.DataFrame(linhas, columns=["limiar", "TP", "FP", "FN", "TN",
                                         "precisao", "recall", "f1", "frac_sinalizada"])


tabela_limiares = varrer_limiares(y_val.to_numpy(), p_val_lr)
print(f"limiares avaliados: {len(tabela_limiares)} (valores distintos de p̂ na validação)")

# ✅ Pontos de verificação pedidos no enunciado
recall_nao_crescente = bool(np.all(np.diff(tabela_limiares["recall"]) <= 0))
frac_no_menor = tabela_limiares["frac_sinalizada"].iloc[0]
print(f"recall não crescente em t: {recall_nao_crescente}")
print(f"frac_sinalizada no menor limiar: {frac_no_menor:.4f}  (deve ser 1,0)")
assert recall_nao_crescente and np.isclose(frac_no_menor, 1.0)

# ✅ RESPOSTA — item 4: os dois limiares
i_f1 = int(tabela_limiares["f1"].idxmax())
t_F1 = tabela_limiares.loc[i_f1, "limiar"]

elegiveis = tabela_limiares[tabela_limiares["recall"] >= 0.70]      # exigência da gerência
i_70 = int(elegiveis["precisao"].idxmax())                          # entre eles, a maior precisão
t_70 = tabela_limiares.loc[i_70, "limiar"]
print(f"\nlimiares com recall >= 0,70: {len(elegiveis)} de {len(tabela_limiares)}")

colunas = ["limiar", "TP", "FP", "FN", "TN", "precisao", "recall", "f1", "frac_sinalizada"]
resumo_limiares = pd.DataFrame([
    tabela_limiares.iloc[(tabela_limiares["limiar"] - 0.5).abs().idxmin()][colunas].rename("t = 0,5 (padrão)"),
    tabela_limiares.loc[i_f1, colunas].rename(f"t_F1 = {t_F1:.4f}"),
    tabela_limiares.loc[i_70, colunas].rename(f"t_70 = {t_70:.4f}"),
])
display(resumo_limiares)

for rotulo, i in [("t_F1", i_f1), ("t_70", i_70)]:
    r = tabela_limiares.loc[i]
    print(f"\nmatriz de confusão em {rotulo} = {r['limiar']:.4f}  (linhas = real, colunas = previsto)")
    print(np.array([[int(r["TN"]), int(r["FP"])], [int(r["FN"]), int(r["TP"])]]))
    print(f"  precisão = {r['precisao']:.4f} | recall = {r['recall']:.4f} | "
          f"sinalizados = {r['frac_sinalizada']:.1%} ({int(r['TP'] + r['FP'])} módulos)")

# ✅ RESPOSTA — item 3: um gráfico com as três métricas em função do limiar
fig, ax = plt.subplots(figsize=(8.4, 4.6))
ax.plot(tabela_limiares["limiar"], tabela_limiares["precisao"], color=COR_LR, lw=2, label="precisão")
ax.plot(tabela_limiares["limiar"], tabela_limiares["recall"], color=COR_ALT, lw=2, label="recall")
ax.plot(tabela_limiares["limiar"], tabela_limiares["f1"], color=COR_3, lw=2, label="F1")
ax.axhline(0.70, color=COR_BASE, ls=":", lw=1.2, label="exigência: recall ≥ 0,70")
ax.axvline(0.5, color="black", ls="-", lw=1, alpha=0.6, label="t = 0,5 (padrão)")
ax.axvline(t_F1, color=COR_3, ls="--", lw=1.4, label=f"t_F1 = {t_F1:.3f}")
ax.axvline(t_70, color=COR_ALT, ls="--", lw=1.4, label=f"t_70 = {t_70:.3f}")
ax.set_xlabel("limiar t")
ax.set_ylabel("valor da métrica")
ax.set_ylim(0, 1.02)
ax.set_xlim(0, 1)
ax.set_title("Precisão, recall e F1 em função do limiar (validação)")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ✅ Verificação do Exercício 3 — execute sem alterar
try:
    tab = varrer_limiares(y_val.to_numpy(), p_val_lr)
    prec_sk, rec_sk, lim_sk = precision_recall_curve(y_val, p_val_lr)
    ref = pd.DataFrame({"limiar": lim_sk, "precisao_sk": prec_sk[:-1], "recall_sk": rec_sk[:-1]})
    comp = ref.merge(tab, on="limiar", how="inner")
    ok = (len(comp) == len(ref)
          and np.allclose(comp["precisao"], comp["precisao_sk"])
          and np.allclose(comp["recall"], comp["recall_sk"]))
    print("OK — sua varredura coincide com precision_recall_curve" if ok else
          "DIFERENTE — confira limiares, a regra (escores >= t) e as fórmulas")
except NotImplementedError as e:
    print("⚠️ Ainda não implementado:", e)

> **Resposta do estudante — item 5:**

---

## ✅ RESPOSTA — Exercício 3, item 5

**Resultados dos dois limiares** (validação: 422 módulos, 65 com defeito):

| | $t$ | TP | FP | FN | TN | precisão | recall | sinalizados |
|:--|--:|--:|--:|--:|--:|--:|--:|--:|
| padrão | 0,5000 | 12 | 11 | 53 | 346 | 0,5217 | 0,1846 | 5,5% (23) |
| $t_{F1}$ | **0,3282** | 28 | 22 | 37 | 335 | 0,5600 | 0,4308 | 11,9% (50) |
| $t_{70}$ | **0,1513** | 46 | 88 | 19 | 269 | 0,3433 | **0,7077** | 31,8% (134) |

**Recomendo $t_{70} = 0{,}1513$ à gerência**, com uma ressalva de capacidade que explico no fim.

**Por quê.** A exigência é explícita: encontrar **pelo menos 70%** dos módulos defeituosos. Isso é uma **restrição**, não um objetivo a negociar — e só 156 dos 282 limiares a satisfazem. Entre eles, $t_{70}$ é o de maior precisão, ou seja, **o que cumpre a exigência desperdiçando o mínimo de horas**. Ele encontra 46 dos 65 defeitos (recall 0,7077) ao preço de 88 alarmes falsos, inspecionando 31,8% da base. Qualquer limiar maior perde a exigência; qualquer limiar menor gera mais alarmes falsos para o mesmo patamar de recall.

**Por que $t_{F1}$ não serve aqui.** O $F_1$ máximo (0,4870, em $t = 0{,}3282$) parece o "melhor ponto do modelo", e é — **para um critério que não é o nosso**. O $F_1$ embute a premissa de que precisão e recall valem **igual** ($\beta = 1$). A gerência disse o contrário: deixar um defeito escapar em um sistema de solo de satélites custa muito mais que uma revisão desnecessária. Com $t_{F1}$, o recall é **0,4308** — isto é, **37 dos 65 defeitos escapam**, quase o dobro dos 19 que escapam em $t_{70}$. Maximizar $F_1$ aqui significaria entregar um detector que **viola o requisito** e justificar a violação com um índice que ninguém pediu. Se eu precisasse de um único índice compatível com a exigência, usaria $F_2$ ou $F_3$, não $F_1$.

**Por que a escolha tem de ser feita na validação, e não no teste.** Porque escolher o limiar **é ajustar um hiperparâmetro**. Percorri 282 limiares e selecionei o melhor segundo um critério: se essa varredura fosse feita no teste, o número reportado seria o **máximo de 282 tentativas** naquela amostra específica, e não uma estimativa do desempenho futuro. O viés não é teórico — a Seção 16.1 mostra que o recall varia de 0,15 a 0,32 **só** por troca de partição, com apenas 65 positivos por conjunto. Com essa variabilidade, o máximo de centenas de candidatos cai sistematicamente acima do valor esperado. O protocolo correto é o da Seção 16.2: escolher o limiar na validação, congelá-lo e medir **uma única vez** no teste — que é o que a investigação final faz.

**A ressalva que eu levaria junto da recomendação.** $t_{70}$ exige inspecionar **31,8%** dos módulos. Se a equipe tem capacidade para 25% (o cenário da investigação final), a exigência de recall ≥ 0,70 e a restrição de capacidade são **incompatíveis** com este modelo: no orçamento de 25% o recall cai para cerca de 0,60. Essa é uma informação que a gerência precisa ter antes de aprovar a política — não é um detalhe técnico, é uma escolha entre contratar mais revisores, aceitar recall menor ou melhorar o modelo. Apresentar $t_{70}$ sem dizer quanto trabalho ele gera seria entregar meia resposta.

---

---
# Parte V — Avaliando o escore: ROC, AUC e curva precisão–recall

## 14. A curva ROC

As métricas das Seções 12 e 13 avaliam **um** detector, isto é, um modelo **com um limiar**. Mas, ao comparar dois modelos, muitas vezes ainda não sabemos qual limiar será usado (a capacidade da equipe muda de uma release para outra). Queremos avaliar a **qualidade da ordenação** que o escore produz, para **todos** os limiares ao mesmo tempo.

### 14.1 Definição

Cada limiar $t$ gera um par (FPR, TPR). A **curva ROC** (*Receiver Operating Characteristic*) é o conjunto desses pontos quando $t$ percorre todos os valores possíveis (Fawcett, 2006):

$$
\mathrm{ROC} = \Big\{ \big(\mathrm{FPR}(t),\ \mathrm{TPR}(t)\big) \;:\; t \in \mathbb{R} \Big\},
\qquad
\mathrm{TPR}(t) = \frac{TP(t)}{P}, \quad \mathrm{FPR}(t) = \frac{FP(t)}{N}.
$$

Propriedades (Fawcett, 2006):

- para $t$ maior que todos os escores, nada é sinalizado: ponto $(0, 0)$; para $t$ menor que todos, tudo é sinalizado: ponto $(1, 1)$;
- a curva é **monótona não decrescente** (Seção 13: TP e FP só crescem quando $t$ diminui);
- a **diagonal** $\mathrm{TPR} = \mathrm{FPR}$ corresponde a sinalizar módulos **ao acaso**: sorteando uma fração $q$ dos módulos, espera-se $\mathrm{TPR} = \mathrm{FPR} = q$;
- um classificador perfeito passa pelo ponto $(0, 1)$: todos os defeitos encontrados sem nenhum alarme falso.

### 14.2 A área sob a curva (AUC) e sua interpretação probabilística

A **AUC** é a área sob a curva ROC, calculada sobre a poligonal pela regra do trapézio (Seção 4.4; Fawcett, 2006, Algoritmo 2):

$$
\mathrm{AUC} = \int_0^1 \mathrm{TPR}\; d(\mathrm{FPR}).
$$

Ela tem uma interpretação muito útil. Sorteie um módulo **com** defeito $X^{+}$ e um módulo **sem** defeito $X^{-}$. Então

$$
\mathrm{AUC} = P\big(s(X^{+}) > s(X^{-})\big) + \tfrac{1}{2}\,P\big(s(X^{+}) = s(X^{-})\big),
$$

isto é, a AUC é a probabilidade de o escore colocar o módulo defeituoso **acima** do não defeituoso, com empates contando metade (Hanley & McNeil, 1982; Fawcett, 2006). Com $P$ positivos e $N$ negativos, essa probabilidade é estimada contando todos os $P \times N$ pares:

$$
\widehat{\mathrm{AUC}} = \frac{1}{P \cdot N} \sum_{i:\, y_i = 1} \;\sum_{j:\, y_j = 0} \Big( \mathbb{I}[s_i > s_j] + \tfrac{1}{2}\,\mathbb{I}[s_i = s_j] \Big),
$$

que é a estatística de Mann–Whitney dividida por $P \cdot N$ (Hanley & McNeil, 1982).

Leitura rápida: AUC $= 0{,}5$ ⇒ ordenação equivalente ao acaso; AUC $= 1$ ⇒ todo positivo recebe escore maior que todo negativo; AUC $< 0{,}5$ ⇒ a ordenação está invertida.

### 14.3 Exemplo manual com 8 módulos

Escores de um detector qualquer, já ordenados do maior para o menor ($P = 4$, $N = 4$):

| módulo | A | B | C | D | E | F | G | H |
|---|---|---|---|---|---|---|---|---|
| escore $s$ | 0,95 | 0,85 | 0,70 | 0,60 | 0,45 | 0,30 | 0,20 | 0,10 |
| real $y$ | 1 | 1 | 0 | 1 | 0 | 0 | 1 | 0 |

Baixando o limiar escore a escore, cada positivo "sobe" a curva em $1/P = 0{,}25$ e cada negativo a "move para a direita" em $1/N = 0{,}25$:

| limiar $t$ | $+\infty$ | 0,95 | 0,85 | 0,70 | 0,60 | 0,45 | 0,30 | 0,20 | 0,10 |
|---|---|---|---|---|---|---|---|---|---|
| TP | 0 | 1 | 2 | 2 | 3 | 3 | 3 | 4 | 4 |
| FP | 0 | 0 | 0 | 1 | 1 | 2 | 3 | 3 | 4 |
| FPR | 0 | 0 | 0 | 0,25 | 0,25 | 0,50 | 0,75 | 0,75 | 1 |
| TPR | 0 | 0,25 | 0,50 | 0,50 | 0,75 | 0,75 | 0,75 | 1 | 1 |

**Área** (só os passos horizontais contribuem, pois nos verticais a largura é zero): $0{,}25 \cdot 0{,}50 + 0{,}25 \cdot 0{,}75 + 0{,}25 \cdot 0{,}75 + 0{,}25 \cdot 1 = 0{,}75$.

**Pares:** A (0,95) e B (0,85) superam os 4 negativos (8 pares); D (0,60) supera E, F, H (3 pares); G (0,20) supera H (1 par). Total: $12/16 = 0{,}75$. As duas contas coincidem.

### 14.4 Algoritmo eficiente

Em vez de reclassificar tudo para cada limiar, basta **ordenar** os exemplos uma vez por escore decrescente e percorrê-los acumulando TP e FP, emitindo um ponto só quando o escore muda. Isso trata corretamente os **empates**: exemplos com o mesmo escore entram juntos, o que gera um segmento diagonal (e é aí que surge o termo $\tfrac{1}{2}$ da AUC). Pseudocódigo adaptado de Fawcett (2006, Algoritmo 1):

```
entrada: rótulos y (P positivos, N negativos), escores s
ordene os índices por s em ordem decrescente
TP ← 0;  FP ← 0;  s_anterior ← +∞;  pontos ← []
para cada exemplo i, na ordem:
    se s_i ≠ s_anterior:
        adicione (FP/N, TP/P) a pontos
        s_anterior ← s_i
    se y_i = 1:  TP ← TP + 1
    senão:       FP ← FP + 1
adicione (FP/N, TP/P) a pontos          # último ponto: (1, 1)
```

Custo: $O(n \log n)$ por causa da ordenação.

Conferindo o exemplo com o scikit-learn:

In [ ]:
s_toy8 = np.array([0.95, 0.85, 0.70, 0.60, 0.45, 0.30, 0.20, 0.10])
y_toy8 = np.array([1,    1,    0,    1,    0,    0,    1,    0])

fpr_toy, tpr_toy, lim_toy = roc_curve(y_toy8, s_toy8, drop_intermediate=False)
display(pd.DataFrame({"limiar": lim_toy, "FPR": fpr_toy, "TPR": tpr_toy}).T.round(2))
print(f"AUC (roc_auc_score): {roc_auc_score(y_toy8, s_toy8):.3f}")

A tabela do `roc_curve` é a mesma construída à mão (o primeiro limiar, `inf`, representa "nada sinalizado"). Por padrão, `roc_curve` usa `drop_intermediate=True`, que remove pontos colineares sem alterar a curva nem a área.

### 14.5 ROC no KC1

In [ ]:
p_val_dummy = dummy.predict_proba(X_val)[:, 1]          # escore constante

fpr_lr, tpr_lr_curva, _ = roc_curve(y_val, p_val_lr)
fpr_dm, tpr_dm_curva, _ = roc_curve(y_val, p_val_dummy)
auc_lr = roc_auc_score(y_val, p_val_lr)
auc_dm = roc_auc_score(y_val, p_val_dummy)

fig, ax = plt.subplots(figsize=(5.6, 5.2))
ax.plot(fpr_lr, tpr_lr_curva, color=COR_LR, lw=2, label=f"Regressão logística (AUC = {auc_lr:.3f})")
ax.plot(fpr_dm, tpr_dm_curva, color=COR_BASE, lw=1.5, ls="--", label=f"Dummy / acaso (AUC = {auc_dm:.3f})")
for t, marcador in [(0.5, "o"), (0.2, "s"), (t_custo, "^")]:
    m = metricas_no_limiar(y_val, p_val_lr, t)
    ax.scatter(m["FP"] / (m["FP"] + m["TN"]), m["recall"], marker=marcador, s=70, color=COR_ALT,
               zorder=3, label=f"ponto de operação t = {t:.3f}")
ax.set_xlabel("taxa de falsos positivos (FPR = 1 − especificidade)")
ax.set_ylabel("taxa de verdadeiros positivos (TPR = recall)")
ax.set_title("Curva ROC na validação")
ax.set_xlim(0, 1); ax.set_ylim(0, 1.01); ax.set_aspect("equal")
ax.legend(loc="lower right", fontsize=8)
plt.show()

### O que acabamos de observar?

- **AUC ≈ 0,80:** sorteando um módulo defeituoso e um não defeituoso da validação, a regressão logística atribui $\hat p$ maior ao defeituoso em cerca de 80% dos pares.
- O dummy dá o mesmo escore a todos os módulos: a curva é a diagonal e a AUC é exatamente 0,5 (todos os pares empatados contam metade).
- Os três **pontos de operação** das Seções 13 e 13.1 estão **sobre** a curva ROC: escolher o limiar é escolher um ponto da curva. O limiar 0,5 fica no canto inferior esquerdo (conservador); o limiar de custo fica bem mais acima e à direita.
- A AUC resume a curva inteira, inclusive regiões que nunca seriam usadas (por exemplo, FPR > 0,8). **Dois modelos com a mesma AUC podem ter desempenhos bem diferentes na região de interesse.**

### 14.6 A AUC só enxerga a ordem dos escores

In [ ]:
z_val_lr = logreg.decision_function(X_val)     # logits: outra escala, mesma ordem que p̂ (σ é crescente)

print(f"AUC usando predict_proba:     {roc_auc_score(y_val, p_val_lr):.6f}")
print(f"AUC usando decision_function: {roc_auc_score(y_val, z_val_lr):.6f}")
print(f"AUC usando p̂³:               {roc_auc_score(y_val, p_val_lr ** 3):.6f}")

### O que acabamos de observar?

As três AUCs são idênticas. Pela interpretação probabilística, a AUC depende apenas das comparações $s_i > s_j$, e qualquer transformação **estritamente crescente** do escore preserva essas comparações. Consequências:

1. **A AUC não avalia calibração.** Um modelo que diz $\hat p = 0{,}9$ para módulos com 30% de chance real de defeito pode ter a mesma AUC de um modelo calibrado. Se você vai usar $\hat p$ como probabilidade (por exemplo, no limiar de custo de Elkan), a AUC não basta.
2. **A AUC não depende da prevalência**, porque TPR e FPR são condicionados à classe real. Isso permite comparar ordenações entre conjuntos de dados, mas esconde o custo operacional quando os positivos são raros (Fawcett, 2006; Saito & Rehmsmeier, 2015). A próxima seção mostra por quê.

### 14.7 Curva precisão–recall e *average precision*

Na validação há 357 módulos sem defeito. Uma FPR de apenas 0,10 já significa ≈ 36 alarmes falsos, mais da metade do total de 65 módulos defeituosos. No gráfico ROC, $\mathrm{FPR} = 0{,}10$ "parece pouco"; na prática, a equipe gasta muito tempo com alarmes falsos.

A **curva precisão–recall (PR)** mostra os pares $\big(\mathrm{Rec}(t), \mathrm{Prec}(t)\big)$. Como a precisão incorpora a prevalência (Seção 12.6), a curva PR revela diretamente esse custo. Duas referências importantes:

- um classificador **aleatório** tem precisão esperada igual a $\pi$ em qualquer recall: a linha de base é a reta horizontal $\mathrm{Prec} = \pi$, e não 0,5 (Saito & Rehmsmeier, 2015);
- o resumo usual é a **average precision (AP)**, uma soma em degraus:

$$
\mathrm{AP} = \sum_{k} (R_k - R_{k-1})\, P_k,
$$

em que $P_k$ e $R_k$ são precisão e recall no $k$-ésimo limiar (scikit-learn, 2026, `average_precision_score`). A AP **não** usa a regra do trapézio, porque a interpolação linear entre pontos no espaço PR é otimista (Davis & Goadrich, 2006).

In [ ]:
prec_lr, rec_lr, _ = precision_recall_curve(y_val, p_val_lr)
ap_lr = average_precision_score(y_val, p_val_lr)

fig, ax = plt.subplots(figsize=(6.4, 4.4))
ax.step(rec_lr, prec_lr, where="post", color=COR_LR, lw=2, label=f"Regressão logística (AP = {ap_lr:.3f})")
ax.axhline(pi_val, color=COR_BASE, ls="--", lw=1.5, label=f"classificador aleatório (precisão = π = {pi_val:.3f})")
for t, marcador in [(0.5, "o"), (0.2, "s"), (t_custo, "^")]:
    m = metricas_no_limiar(y_val, p_val_lr, t)
    ax.scatter(m["recall"], m["precisão"], marker=marcador, s=70, color=COR_ALT, zorder=3, label=f"t = {t:.3f}")
ax.set_xlabel("recall")
ax.set_ylabel("precisão")
ax.set_xlim(0, 1); ax.set_ylim(0, 1.01)
ax.set_title("Curva precisão–recall na validação")
ax.legend(loc="upper right", fontsize=8)
plt.show()

### O que acabamos de observar?

- A AP ≈ 0,44 é quase **três vezes** a linha de base $\pi \approx 0{,}15$: a ordenação é muito melhor que o acaso.
- Mas a curva PR mostra o preço operacional que a ROC suaviza: para encontrar **70% ou mais** dos defeitos, a precisão fica entre ≈ 0,28 e 0,35, ou seja, cerca de **2 em cada 3 módulos inspecionados não têm defeito**.
- A curva PR é "serrilhada" em recall baixo: com poucos positivos, cada TP ou FP muda bastante a precisão.

**Recomendação prática:** em problemas desbalanceados, reporte **ROC/AUC e PR/AP** juntas, sempre com a linha de base de cada uma (Saito & Rehmsmeier, 2015).

### ✅ Checkpoint 3

1. O que um ponto da curva ROC representa? E a curva inteira?
2. Por que a AUC do dummy é exatamente 0,5?
3. Explique com suas palavras a frase "a AUC é a probabilidade de um positivo ficar acima de um negativo".
4. Por que a linha de base da curva PR é $\pi$ e não 0,5?
5. Por que a AUC de `predict_proba` e de `decision_function` coincidem? Que informação a AUC ignora?

---
## Exercício 4 — ROC e AUC do zero, e um detector de uma variável só

**Objetivo:** implementar a curva ROC e a AUC a partir das definições, validar a implementação (inclusive com empates) e usá-la para testar uma hipótese sobre o valor da informação multivariada.

**Tarefa:**

1. Implemente em NumPy (sem `sklearn.metrics`):
   - `curva_roc(y_true, escores)` → arrays `fpr` e `tpr`, começando em $(0, 0)$ e terminando em $(1, 1)$, seguindo o pseudocódigo da Seção 14.4 (**com tratamento de empates**);
   - `auc_trapezio(fpr, tpr)` → área pela regra do trapézio (Seção 4.4);
   - `auc_pares(y_true, escores)` → estimativa pela contagem de pares da Seção 14.2 (empates valem ½). *Dica:* com vetores de escores dos positivos e dos negativos, a comparação de todos os pares cabe em uma única operação com *broadcasting*.
2. Execute a **célula de verificação**. Ela testa suas funções no exemplo de 8 módulos, nos escores da regressão logística e na coluna `loc` da validação (que tem **muitos empates**).
3. **Hipótese antes do experimento.** Considere usar **uma única métrica de código** como escore, sem modelo algum: quanto maior `loc` (ou `v(g)`), mais suspeito o módulo. Registre na célula de resposta sua hipótese: a AUC de `loc` sozinha será muito menor, parecida ou maior que a da regressão logística com 21 atributos? E a de `v(g)`?
4. Com **suas** funções, calcule a AUC de `loc`, `v(g)` e da regressão logística na validação e faça **um** gráfico com as três curvas ROC e a diagonal.
5. Interprete em 4 a 6 frases: a hipótese foi sustentada? O que o resultado sugere sobre a informação que a regressão logística extrai dos 21 atributos? A AUC é suficiente para decidir entre a regressão logística e o detector "ordenar por `loc`"? Considere, por exemplo, o que acontece nos pontos de operação usados na Seção 13 e o que cada abordagem exige para ser aplicada.

**Entregável:** três funções, verificação executada, hipótese registrada **antes** do experimento, gráfico e interpretação.

> **Resposta do estudante — item 3 (hipótese, escrita antes do experimento):**

---

## ✅ RESPOSTA — Exercício 4, item 3 (hipótese registrada ANTES de rodar)

**Minha hipótese: a AUC de `loc` sozinha será claramente menor que a da regressão logística — algo em torno de 0,70 a 0,73, contra ≈ 0,80 do modelo com 21 atributos. Para `v(g)` espero um pouco menos, perto de 0,68 a 0,72.**

**Raciocínio.**

1. **O modelo tem mais informação disponível.** A regressão logística combina 21 atributos com pesos ajustados por máxima verossimilhança; `loc` é uma única coluna. Se os atributos trouxessem sinal complementar — tamanho, complexidade de fluxo, vocabulário de operadores são conceitos distintos —, a combinação deveria ordenar melhor que qualquer um deles isolado;
2. **o histograma da Seção 6.3 mostra sobreposição grande** entre as classes em `loc`: há muitos módulos grandes sem defeito e pequenos com defeito. Isso limita o quanto `loc` sozinho consegue ordenar;
3. **`loc` tem muitos empates** (é um inteiro, e há centenas de módulos minúsculos — a Seção 6.4 mostrou 1.192 vetores distintos em 2.109 linhas). Empates contam ½ na AUC, o que **puxa a AUC de `loc` para baixo** mecanicamente, por um motivo que nada tem a ver com a qualidade da métrica;
4. **`v(g)` deve ficar atrás de `loc`** porque é mais grosseiro: a maioria dos módulos tem complexidade ciclomática baixa e repetida (1, 2, 3…), concentrando ainda mais empates, e a complexidade ignora completamente o tamanho do módulo.

**Previsão numérica:** AUC(`loc`) $\approx 0{,}72$; AUC(`v(g)`) $\approx 0{,}70$; AUC(LogReg) $\approx 0{,}80$. Espero uma diferença de **8 a 10 pontos** em favor do modelo — margem suficiente para justificar a complexidade de treinar e manter uma regressão logística em vez de ordenar uma planilha por uma coluna.

---

In [ ]:
# Exercício 4 — itens 1 e 4

# ✅ RESPOSTA — item 1a: curva ROC pelo algoritmo da Seção 14.4 (com empates)
def curva_roc(y_true, escores):
    """Pontos (FPR, TPR) de (0,0) a (1,1), emitindo um ponto por valor distinto de escore.

    Exemplos com o MESMO escore entram juntos (um único ponto), o que produz o
    segmento diagonal correspondente aos empates.
    """
    y_true = np.asarray(y_true)
    escores = np.asarray(escores)
    P = int((y_true == 1).sum())
    N = int((y_true == 0).sum())

    ordem = np.argsort(-escores, kind="mergesort")      # escore decrescente, estável
    y_ord, s_ord = y_true[ordem], escores[ordem]

    tp = fp = 0
    s_anterior = np.inf
    fpr, tpr = [], []
    for y_i, s_i in zip(y_ord, s_ord):
        if s_i != s_anterior:                           # mudou o escore: fecha um ponto
            fpr.append(fp / N)
            tpr.append(tp / P)
            s_anterior = s_i
        if y_i == 1:
            tp += 1
        else:
            fp += 1
    fpr.append(fp / N)                                  # último ponto: (1, 1)
    tpr.append(tp / P)
    return np.array(fpr), np.array(tpr)


# ✅ RESPOSTA — item 1b: área pela regra do trapézio (Seção 4.4)
def auc_trapezio(fpr, tpr):
    fpr, tpr = np.asarray(fpr), np.asarray(tpr)
    return float(np.sum(np.diff(fpr) * (tpr[1:] + tpr[:-1]) / 2.0))


# ✅ RESPOSTA — item 1c: contagem de pares (Seção 14.2), empates valendo ½
def auc_pares(y_true, escores):
    y_true, escores = np.asarray(y_true), np.asarray(escores)
    s_pos = escores[y_true == 1]
    s_neg = escores[y_true == 0]
    diferenca = s_pos[:, None] - s_neg[None, :]         # broadcasting: todos os P×N pares
    vitorias = np.sum(diferenca > 0) + 0.5 * np.sum(diferenca == 0)
    return float(vitorias / (s_pos.size * s_neg.size))


# ✅ RESPOSTA — item 4: AUC de loc, v(g) e da regressão logística, com MINHAS funções
escores_candidatos = {
    "loc (linhas de código)": X_val["loc"].to_numpy().astype(float),
    "v(g) (compl. ciclomática)": X_val["v(g)"].to_numpy().astype(float),
    "Regressão logística (21 atributos)": p_val_lr,
}

print(f"{'escore':<36} {'AUC (trapézio)':>15} {'AUC (pares)':>13} {'empates':>9}")
print("-" * 76)
curvas = {}
for nome, s in escores_candidatos.items():
    f, t = curva_roc(y_val.to_numpy(), s)
    curvas[nome] = (f, t)
    n_distintos = len(np.unique(s))
    print(f"{nome:<36} {auc_trapezio(f, t):>15.4f} {auc_pares(y_val.to_numpy(), s):>13.4f} "
          f"{1 - n_distintos / len(s):>8.1%}")

fig, ax = plt.subplots(figsize=(5.8, 5.4))
for (nome, (f, t)), cor in zip(curvas.items(), [COR_3, COR_ALT, COR_LR]):
    ax.plot(f, t, lw=2, color=cor, label=f"{nome}\nAUC = {auc_trapezio(f, t):.4f}")
ax.plot([0, 1], [0, 1], ls="--", lw=1.2, color=COR_BASE, label="acaso (AUC = 0,5)")
ax.set_xlabel("FPR = 1 − especificidade")
ax.set_ylabel("TPR = recall")
ax.set_xlim(0, 1); ax.set_ylim(0, 1.01); ax.set_aspect("equal")
ax.set_title("ROC: um modelo de 21 atributos contra duas métricas isoladas")
ax.legend(loc="lower right", fontsize=7)
plt.tight_layout()
plt.show()

# A AUC resume a curva inteira; a região útil é a de FPR baixo. Comparação local:
print("\nTPR na região operacional (FPR ≈ 0,20, perto do orçamento de 25%):")
for nome, (f, t) in curvas.items():
    print(f"  {nome:<36} TPR ≈ {np.interp(0.20, f, t):.4f}")
print(f"\nAP (average precision) — sensível à prevalência, π = {y_val.mean():.4f}:")
for nome, s in escores_candidatos.items():
    print(f"  {nome:<36} AP = {average_precision_score(y_val, s):.4f}")

In [ ]:
# ✅ Verificação do Exercício 4 — execute sem alterar
try:
    casos = {
        "exemplo de 8 módulos": (y_toy8, s_toy8),
        "LogReg (validação)": (y_val.to_numpy(), p_val_lr),
        "loc (validação, com empates)": (y_val.to_numpy(), X_val["loc"].to_numpy()),
    }
    for nome, (yt, s) in casos.items():
        fpr_e, tpr_e = curva_roc(yt, s)
        ref = roc_auc_score(yt, s)
        ok_extremos = np.allclose([fpr_e[0], tpr_e[0], fpr_e[-1], tpr_e[-1]], [0, 0, 1, 1])
        ok_monotona = np.all(np.diff(fpr_e) >= 0) and np.all(np.diff(tpr_e) >= 0)
        ok_trap = np.isclose(auc_trapezio(fpr_e, tpr_e), ref)
        ok_pares = np.isclose(auc_pares(yt, s), ref)
        tudo = ok_extremos and ok_monotona and ok_trap and ok_pares
        print(f"{nome:30s} extremos: {ok_extremos} | monótona: {ok_monotona} | "
              f"AUC trapézio: {ok_trap} | AUC pares: {ok_pares} → {'OK' if tudo else 'revise'}")
except NotImplementedError as e:
    print("⚠️ Ainda não implementado:", e)

> **Resposta do estudante — item 5 (interpretação):**

---

## ✅ RESPOSTA — Exercício 4, item 5 (interpretação)

**Resultados medidos com as minhas funções** (validação, 422 módulos, 65 com defeito):

| escore | AUC (trapézio) | AUC (pares) | AP | TPR em FPR ≈ 0,20 |
|:--|--:|--:|--:|--:|
| `loc` sozinho | **0,8018** | 0,8018 | 0,4048 | 0,5538 |
| `v(g)` sozinho | 0,7195 | 0,7195 | 0,2993 | 0,5517 |
| Regressão logística (21 atributos) | **0,8034** | 0,8034 | **0,4378** | **0,6154** |

**A hipótese NÃO se sustentou — e errou pelo motivo mais interessante possível.** Eu previ 8 a 10 pontos de vantagem para o modelo; a diferença real é de **0,0016** de AUC. Ordenar os módulos por **linhas de código**, uma única coluna que qualquer script extrai em um segundo, ordena **tão bem** quanto uma regressão logística ajustada com 21 métricas de McCabe e Halstead. A previsão para `v(g)` foi a única correta em direção (0,7195, atrás de `loc`), e também errei a magnitude dos empates: `loc` tem **80,3%** de escores repetidos e **mesmo assim** lidera as métricas isoladas.

**O que isso sugere sobre a informação que o modelo extrai.** Praticamente nada além de tamanho. As 21 métricas são quase todas **funções monotônicas do tamanho do módulo** — `n`, `v`, `e`, `total_Op`, `total_Opnd`, `branchCount` crescem todas junto com `loc`, e a Seção 10.1 já avisava que a correlação entre `v(g)` e `branchCount` é ≈ 0,999. Um conjunto de 21 colunas que mede 21 vezes a mesma coisa carrega a informação de **uma** coluna. A regressão logística não tem de onde tirar sinal complementar: ela redistribui pesos entre atributos colineares (daí os coeficientes instáveis e de sinais opostos mencionados na Seção 10.1) e chega ao mesmo lugar. Esse resultado reproduz, em miniatura, uma crítica conhecida da literatura de predição de defeitos — a de que grande parte do poder preditivo das métricas estáticas se reduz a "módulos maiores têm mais defeitos".

**A AUC é suficiente para decidir entre os dois? Não.** Ela é o começo da conversa, não o fim, por quatro razões concretas:

1. **a diferença está dentro do ruído.** A Seção 16.1 mediu AUC variando de ≈ 0,75 a ≈ 0,85 entre partições com o **mesmo** modelo. Uma diferença de 0,0016 em uma única partição não é evidência de nada;
2. **a AUC resume a curva inteira, inclusive a parte inútil — e aqui ela esconde o resultado que interessa.** A decisão real acontece perto do orçamento da equipe, isto é, em FPR baixo. Nessa região (FPR ≈ 0,20) o modelo entrega TPR $= 0{,}6154$ contra $0{,}5538$ do `loc`: **6 pontos percentuais de recall** que a diferença de AUC (0,0016) simplesmente não mostra. O contraexemplo complementar está na mesma tabela: `v(g)` tem AUC bem pior (0,7195) e, ainda assim, TPR $= 0{,}5517$ nessa região — praticamente igual ao `loc`. Ou seja, **a AUC não localiza a vantagem**: curvas podem se cruzar, e o que decide é a região em que o sistema vai operar;
3. **a AUC ignora calibração** (Seção 14.6). `loc` **não é uma probabilidade**: dá para ordenar por ele, mas não dá para aplicar o limiar de custo de Elkan ($t^\star = c_{FP}/(c_{FP}+c_{FN})$), que exige $\hat p \approx P(\text{defeito}\mid\mathbf x)$. A LogReg entrega escores calibrados; o `loc` cru, não. Para a política de inspeção por **orçamento** isso não importa (basta a ordem); para uma política por **custo**, importa muito;
4. **a AP distingue o que a AUC esconde.** 0,4378 contra 0,4048 — o modelo é melhor por ≈ 3 pontos na métrica que incorpora a prevalência, isto é, na que reflete o esforço desperdiçado. É uma vantagem real, ainda que modesta.

**O que cada abordagem exige para ser aplicada** — e aqui o `loc` ganha pontos que a AUC não mostra: ele não precisa de treino, de padronização, de conjunto rotulado, de versionamento de modelo nem de retreino quando o projeto muda; é uma ordenação de planilha, auditável por qualquer engenheiro e imune a *data leakage*. A regressão logística exige histórico rotulado de defeitos (caro de obter), um `Pipeline` mantido em produção e revalidação periódica.

**Conclusão honesta:** com estes dados, `loc` é uma **linha de base dura de bater** e deve entrar como candidato formal na investigação final — não como piada, mas como concorrente sério. Eu só justificaria a regressão logística se (i) a vantagem se mantivesse ao longo de várias partições, (ii) ela aparecesse na região operacional de interesse e (iii) a calibração fosse necessária para a política escolhida. A condição (ii) **já se verifica** (6 pontos em FPR ≈ 0,20); é exatamente (i) e (iii) que a investigação final vai testar.

---

---
## 15. Mover o limiar ou reponderar as classes?

Uma alternativa muito usada para lidar com desbalanceamento é **mudar a função objetivo**, dando mais peso aos exemplos da classe rara. Na `LogisticRegression`, isso é feito com `class_weight`, que define os pesos $s_i$ da função objetivo da Seção 7.2. A opção `class_weight="balanced"` usa, para a classe $c$,

$$
s_c = \frac{n}{K \cdot n_c},
$$

em que $n$ é o número de exemplos de treino, $K = 2$ é o número de classes e $n_c$ é o número de exemplos da classe $c$ (scikit-learn, 2026, `LogisticRegression`). No nosso treino, $s_1 = 1265/(2 \cdot 196) \approx 3{,}23$ e $s_0 = 1265/(2 \cdot 1069) \approx 0{,}59$: cada módulo defeituoso pesa cerca de 5,5 vezes um módulo sem defeito na entropia cruzada.

Pelo mesmo argumento da Seção 13 (derivada em relação ao intercepto igual a zero), no ótimo vale $\sum_i s_i\,(\hat p_i - y_i) = 0$: a média **ponderada** de $\hat p$ no treino passa a ser $\sum_i s_i y_i / \sum_i s_i = 0{,}5$, e não mais $\pi$.

> **Pare e pense (não precisa entregar):** com `class_weight="balanced"` e limiar 0,5, o que você espera para recall, precisão e acurácia? E para a AUC?

In [ ]:
logreg_bal = make_pipeline(StandardScaler(),
                           LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced"))
logreg_bal.fit(X_treino, y_treino)
p_val_bal = logreg_bal.predict_proba(X_val)[:, 1]
yhat_val_bal = logreg_bal.predict(X_val)

# Pesos "balanced" e média ponderada de p̂ no treino
n_tr, n1_tr = len(y_treino), y_treino.sum()
pesos = np.where(y_treino == 1, n_tr / (2 * n1_tr), n_tr / (2 * (n_tr - n1_tr)))
p_treino_bal = logreg_bal.predict_proba(X_treino)[:, 1]
print(f"Pesos: s_1 = {pesos.max():.3f}, s_0 = {pesos.min():.3f}")
print(f"Média ponderada de p̂ (balanced) no treino: {np.sum(pesos * p_treino_bal) / pesos.sum():.4f}")

comparacao_cw = tabela_metricas({"LogReg (limiar 0,5)": yhat_val_lr,
                                 "LogReg balanced (limiar 0,5)": yhat_val_bal}, y_val)
comparacao_cw["AUC"] = [roc_auc_score(y_val, p_val_lr), roc_auc_score(y_val, p_val_bal)]
comparacao_cw["AP"] = [average_precision_score(y_val, p_val_lr), average_precision_score(y_val, p_val_bal)]
comparacao_cw["média de p̂"] = [p_val_lr.mean(), p_val_bal.mean()]
display(comparacao_cw)

rho = pd.Series(p_val_lr).corr(pd.Series(p_val_bal), method="spearman")
print(f"Correlação de postos (Spearman) entre os escores dos dois modelos: {rho:.3f}")

### O que acabamos de observar?

- Com `class_weight="balanced"`, o recall sobe de ≈ 0,18 para ≈ 0,71, a precisão cai para ≈ 0,35 e a acurácia **cai** (≈ 0,75) — mais uma vez, a acurácia penaliza o detector que de fato encontra defeitos.
- A **AUC praticamente não muda**, e a correlação de postos entre os escores é ≈ 0,995: os dois modelos **ordenam os módulos quase da mesma forma**. A reponderação desloca os escores para cima (média de $\hat p$ ≈ 0,39 contra ≈ 0,15), o que tem efeito parecido com **baixar o limiar** do modelo original.
- O preço: $\hat p$ do modelo ponderado **deixa de estimar** $P(\text{defeito} \mid \mathbf{x})$ na população real. Usar o limiar de custo de Elkan (Seção 13.1) sobre esses escores seria um erro.

Vamos testar a equivalência: ajustamos o limiar do modelo **sem** pesos para sinalizar a mesma fração de módulos que o modelo balanceado sinaliza com limiar 0,5.

In [ ]:
frac_bal = yhat_val_bal.mean()
t_equiv = limiar_por_orcamento(p_val_lr, frac_bal)
yhat_val_equiv = (p_val_lr >= t_equiv).astype(int)

tabela_metricas({"LogReg balanced (limiar 0,5)": yhat_val_bal,
                 f"LogReg sem pesos (limiar {t_equiv:.3f})": yhat_val_equiv}, y_val)

### O que acabamos de observar?

Sinalizando praticamente a mesma fração de módulos, os dois detectores têm recall e precisão muito próximos (diferenças de um ou dois módulos). Neste problema, **reponderar as classes e mover o limiar são duas formas de chegar a pontos de operação semelhantes** sobre curvas ROC quase idênticas. Mover o limiar tem duas vantagens: não exige reajustar o modelo e preserva a interpretação de $\hat p$ como probabilidade. A reponderação pode ser mais vantajosa em modelos cuja ordenação muda com os pesos (por exemplo, árvores e *ensembles*), o que você poderá investigar nas próximas aulas.

---
## 16. Armadilhas: variabilidade e uso do conjunto de teste

### 16.1 Quanto as métricas variam de uma partição para outra?

Na Seção 8 vimos que validação e teste têm apenas 65 módulos defeituosos. Vamos repetir **todo** o procedimento (divisão estratificada 60/20/20, ajuste e avaliação na validação) com 10 sementes diferentes, mantendo o modelo e o limiar fixos.

In [ ]:
resultados = []
for semente in range(10):
    X_tv_s, _, y_tv_s, _ = train_test_split(X, y, test_size=0.20, stratify=y, random_state=semente)
    X_tr_s, X_va_s, y_tr_s, y_va_s = train_test_split(X_tv_s, y_tv_s, test_size=0.25,
                                                      stratify=y_tv_s, random_state=semente)
    modelo = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000)).fit(X_tr_s, y_tr_s)
    escores = modelo.predict_proba(X_va_s)[:, 1]
    yhat = (escores >= 0.5).astype(int)
    resultados.append({"semente": semente,
                       "acurácia": accuracy_score(y_va_s, yhat),
                       "precisão": precision_score(y_va_s, yhat, zero_division=0),
                       "recall": recall_score(y_va_s, yhat),
                       "F1": f1_score(y_va_s, yhat),
                       "AUC": roc_auc_score(y_va_s, escores)})

variab = pd.DataFrame(resultados).set_index("semente")
display(variab.agg(["mean", "std", "min", "max"]).T)

fig, ax = plt.subplots(figsize=(7, 3.6))
for k, col in enumerate(variab.columns):
    ax.scatter(np.full(len(variab), k), variab[col], color=COR_LR, alpha=0.7, s=30)
    ax.hlines(variab[col].mean(), k - 0.25, k + 0.25, color=COR_ALT, lw=2)
ax.set_xticks(range(len(variab.columns)), variab.columns)
ax.set_ylabel("valor na validação")
ax.set_ylim(0, 1)
ax.set_title("Mesmo modelo, 10 partições aleatórias (traço laranja = média)")
plt.show()

### O que acabamos de observar?

- A **acurácia** quase não varia (desvio-padrão < 0,01): ela é dominada pelos ≈ 357 negativos da validação.
- **Recall, precisão e F1** variam muito (o recall vai de ≈ 0,15 a ≈ 0,32): dependem de poucas dezenas de positivos.
- A **AUC** varia de ≈ 0,75 a ≈ 0,85 entre partições.

Consequência direta: **uma diferença de 0,02 de AUC entre dois modelos em uma única partição não é evidência de que um seja melhor.** Antes de concluir, compare a diferença com a variabilidade. A ferramenta sistemática para isso é a **validação cruzada**, tema de uma aula futura; na investigação final você fará uma versão simples, repetindo partições.

### 16.2 O protocolo de uso do teste

1. Todas as decisões — escolha do modelo, de hiperparâmetros como `C` e `class_weight`, de transformações e do **limiar** — são tomadas com treino e validação.
2. Com tudo decidido, avalia-se **uma única vez** no teste, usando o modelo ajustado no treino e o limiar escolhido na validação.
3. O resultado do teste é **reportado**, não usado para voltar e ajustar decisões. Se isso acontecer, o teste vira validação e a estimativa final fica otimista.

Exemplos de **vazamento** relacionados a esta aula:

- escolher o limiar que maximiza o F1 **no teste**;
- escolher, entre vários modelos, o de maior AUC **no teste** e reportar essa mesma AUC;
- ajustar o `StandardScaler` com **todos** os dados antes da divisão (o `Pipeline` evita esse erro);
- ignorar que cópias exatas de módulos podem estar no treino e no teste (Seção 6.4).

---
## Investigação final — Política de inspeção de código para a próxima release

### Problema

A equipe de qualidade do KC1 tem capacidade para inspecionar **no máximo 25% dos módulos** de uma release. A gerência quer (i) **encontrar o maior número possível de módulos defeituosos** dentro desse orçamento e (ii) uma **estimativa honesta** do que esperar da política adotada, incluindo sua incerteza. Você deve propor essa política.

### Objetivo

Escolher um **detector** (modelo ou regra + limiar) que respeite o orçamento, justificar a escolha com evidências obtidas **sem usar o teste** e, ao final, estimar no teste o desempenho esperado.

### Restrições

- Use a divisão da Seção 8 (`X_treino`, `X_val`, `X_teste`). **O teste é usado uma única vez**, depois de todas as decisões.
- O limiar de cada candidato deve ser escolhido na validação, de forma que a fração de módulos sinalizados na validação seja **≤ 25%**.
- Compare **pelo menos quatro** candidatos:
  1. uma linha de base **sem aprendizado**, que ordena os módulos por uma única métrica de código (por exemplo, `loc`);
  2. a regressão logística da Seção 10;
  3. e 4. pelo menos **duas variações** estudadas neste notebook, por exemplo: `class_weight="balanced"`; transformação `FunctionTransformer(np.log1p)` antes do `StandardScaler` (Menzies, Greenwald & Frank, 2007); outro valor de `C`.
- Não use conteúdo não visto nesta aula (validação cruzada, outros algoritmos, técnicas de reamostragem de classes como *oversampling*).

### Tarefas

1. **Métrica principal.** Defina a métrica principal da decisão e as métricas secundárias que você vai reportar. Justifique a partir do problema (e não por hábito).
2. **Comparação na validação.** Para cada candidato: ajuste no treino (quando houver ajuste), obtenha os escores da validação, escolha o limiar pelo orçamento e registre as métricas. Reporte também AUC e AP de cada candidato.
3. **Robustez.** Repita a comparação em **pelo menos 5 partições aleatórias diferentes** (60/20/20 estratificado, como na Seção 16.1) e reporte média e desvio-padrão da métrica principal de cada candidato. As diferenças entre candidatos são maiores que a variabilidade entre partições?
4. **Decisão.** Escolha o detector final (candidato + limiar da partição original). Se as evidências não distinguirem os candidatos, use um critério secundário explícito (simplicidade, custo de manutenção, interpretabilidade, dependência de calibração...).
5. **Avaliação final no teste (uma vez).** Aplique o detector escolhido a `X_teste`: matriz de confusão, recall, precisão, FPR e fração sinalizada. Compare com a validação e comente as diferenças à luz da Seção 16.1.
6. **Custo.** Com $c_{FP} = 1$ hora e $c_{FN} = 10$ horas, calcule o custo total no teste da sua política e compare com as políticas "não inspecionar nada" e "inspecionar tudo" (ignorando o orçamento). A sua política é a de menor custo? Se não for, isso invalida a escolha? Discuta.

### Evidências necessárias

- tabela de comparação na validação (partição original);
- tabela ou gráfico da variabilidade entre partições;
- matriz de confusão e métricas no teste;
- tabela de custos das três políticas.

### Conclusão

Escreva uma conclusão técnica em Markdown (2 a 4 parágrafos), dirigida à gerência de qualidade, respondendo: qual política adotar; quantos defeitos esperar encontrar e com qual incerteza; quanto esforço será desperdiçado em alarmes falsos; e quais **limitações** afetam essas estimativas (considere, no mínimo, a variabilidade entre partições, os módulos duplicados da Seção 6.4 e a aplicação do detector a um projeto com prevalência diferente — Seção 12.6).

### Critérios de avaliação

| Critério | Peso |
|---|---|
| Correção metodológica: limiares e escolhas feitos na validação, teste usado uma única vez, pré-processamento dentro do `Pipeline` | 30% |
| Implementação e resultados: candidatos corretos, orçamento respeitado, métricas corretas | 25% |
| Análise de robustez: variabilidade entre partições usada na decisão | 15% |
| Interpretação, custos e conclusão: argumentação técnica, limitações e clareza para a gerência | 30% |

In [ ]:
# 🔬 Investigação final — Tarefas 1 e 2: candidatos e comparação na validação (partição original)

ORCAMENTO = 0.25

# ✅ RESPOSTA — Tarefa 2
def construir_candidatos(X_fit, y_fit):
    """Ajusta os candidatos em (X_fit, y_fit) e devolve {nome: função de escore}.

    A linha de base por métrica de código não tem fit: o escore é a própria coluna.
    Todo pré-processamento vive DENTRO do Pipeline, de modo que o StandardScaler
    nunca vê dados fora do treino.
    """
    lr = make_pipeline(StandardScaler(),
                       LogisticRegression(C=1.0, max_iter=1000)).fit(X_fit, y_fit)
    lr_bal = make_pipeline(StandardScaler(),
                           LogisticRegression(C=1.0, max_iter=1000,
                                              class_weight="balanced")).fit(X_fit, y_fit)
    lr_log = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(),
                           LogisticRegression(C=1.0, max_iter=1000)).fit(X_fit, y_fit)
    lr_c01 = make_pipeline(StandardScaler(),
                           LogisticRegression(C=0.1, max_iter=1000)).fit(X_fit, y_fit)
    return {
        "1. baseline: ordenar por loc": lambda Xq: Xq["loc"].to_numpy().astype(float),
        "2. LogReg (C=1)":              lambda Xq: lr.predict_proba(Xq)[:, 1],
        "3. LogReg balanced":           lambda Xq: lr_bal.predict_proba(Xq)[:, 1],
        "4. LogReg log1p":              lambda Xq: lr_log.predict_proba(Xq)[:, 1],
        "5. LogReg C=0,1":              lambda Xq: lr_c01.predict_proba(Xq)[:, 1],
    }


def avaliar_no_orcamento(y_true, escores, orcamento=ORCAMENTO):
    """Escolhe o limiar pelo orçamento e devolve as métricas do ponto de operação."""
    t = limiar_por_orcamento(escores, orcamento)
    yhat = (escores >= t).astype(int)
    (tn, fp), (fn, tp) = matriz_confusao(y_true, yhat)
    return {"limiar": t, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "recall": tp / (tp + fn), "precisão": tp / (tp + fp) if (tp + fp) > 0 else 0.0,
            "FPR": fp / (fp + tn), "% sinalizados": yhat.mean(),
            "AUC": roc_auc_score(y_true, escores),
            "AP": average_precision_score(y_true, escores)}


escoradores = construir_candidatos(X_treino, y_treino)
comparacao_val = pd.DataFrame(
    {nome: avaliar_no_orcamento(y_val, f(X_val)) for nome, f in escoradores.items()}).T

print(f"Orçamento de inspeção: {ORCAMENTO:.0%} dos módulos  |  validação: {len(y_val)} módulos, "
      f"{int(y_val.sum())} com defeito")
print(f"Métrica principal: RECALL no orçamento (fração dos defeitos encontrada inspecionando "
      f"≤ {ORCAMENTO:.0%} da base)\n")
display(comparacao_val.round(4))

print(f"referência — inspeção ALEATÓRIA de {ORCAMENTO:.0%}: recall esperado ≈ {ORCAMENTO:.2f}")
print(f"orçamento respeitado por todos: {bool((comparacao_val['% sinalizados'] <= ORCAMENTO).all())}")
print(f"melhor recall na validação: {comparacao_val['recall'].idxmax()} "
      f"({comparacao_val['recall'].max():.4f})")
print(f"amplitude entre o melhor e o pior candidato: "
      f"{comparacao_val['recall'].max() - comparacao_val['recall'].min():.4f}")

In [ ]:
# 🔬 Investigação final — Tarefa 3: robustez em pelo menos 5 partições

# ✅ RESPOSTA — Tarefa 3: 10 partições 60/20/20 estratificadas (como na Seção 16.1).
# Todo o procedimento é repetido: divisão -> fit no treino -> escores da validação ->
# limiar pelo orçamento -> recall. Nenhuma partição toca o conjunto de teste original.
N_PARTICOES = 10
recalls = {nome: [] for nome in escoradores}

for semente in range(N_PARTICOES):
    Xtv_s, _, ytv_s, _ = train_test_split(X, y, test_size=0.20, stratify=y, random_state=semente)
    Xtr_s, Xva_s, ytr_s, yva_s = train_test_split(Xtv_s, ytv_s, test_size=0.25,
                                                  stratify=ytv_s, random_state=semente)
    for nome, f in construir_candidatos(Xtr_s, ytr_s).items():
        recalls[nome].append(avaliar_no_orcamento(yva_s, f(Xva_s))["recall"])

robustez = pd.DataFrame(recalls)
resumo_robustez = pd.DataFrame({
    "média": robustez.mean(),
    "desvio-padrão": robustez.std(ddof=1),
    "mínimo": robustez.min(),
    "máximo": robustez.max(),
    "val. original": comparacao_val["recall"],
})
print(f"Recall no orçamento de {ORCAMENTO:.0%} — {N_PARTICOES} partições aleatórias\n")
display(resumo_robustez.round(4))

maior_dif = resumo_robustez["média"].max() - resumo_robustez["média"].min()
desvio_tipico = resumo_robustez["desvio-padrão"].mean()
print(f"maior diferença entre MÉDIAS de candidatos: {maior_dif:.4f}")
print(f"desvio-padrão típico ENTRE PARTIÇÕES:       {desvio_tipico:.4f}")
print(f"\n→ a variabilidade entre partições é {desvio_tipico / maior_dif:.1f}x a maior diferença")
print("  entre candidatos: as evidências NÃO distinguem os candidatos aprendidos.")

fig, ax = plt.subplots(figsize=(8.6, 4.2))
for k, nome in enumerate(robustez.columns):
    ax.scatter(np.full(N_PARTICOES, k), robustez[nome], color=COR_LR, alpha=0.65, s=34)
    ax.hlines(robustez[nome].mean(), k - 0.26, k + 0.26, color=COR_ALT, lw=2.5)
ax.axhline(ORCAMENTO, color=COR_BASE, ls=":", lw=1.4, label=f"inspeção aleatória ({ORCAMENTO:.0%})")
ax.set_xticks(range(len(robustez.columns)))
ax.set_xticklabels([n.split(". ")[1] for n in robustez.columns], rotation=20, ha="right", fontsize=8)
ax.set_ylabel(f"recall no orçamento de {ORCAMENTO:.0%}")
ax.set_ylim(0, 0.85)
ax.set_title(f"Mesma política, {N_PARTICOES} partições (traço laranja = média)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# 🔬 Investigação final — Tarefas 5 e 6: avaliação única no teste e análise de custos

# ✅ RESPOSTA — Tarefa 5. Detector escolhido na Tarefa 4: candidato 2 (LogReg C=1),
# ajustado em X_treino, com o limiar escolhido na VALIDAÇÃO pelo orçamento de 25%.
CANDIDATO_ESCOLHIDO = "2. LogReg (C=1)"
escore = escoradores[CANDIDATO_ESCOLHIDO]

t_final = limiar_por_orcamento(escore(X_val), ORCAMENTO)   # decidido só com a validação
print(f"detector: {CANDIDATO_ESCOLHIDO}  |  limiar congelado na validação: t = {t_final:.4f}")
print("=" * 78)
print(">>> PRIMEIRO E ÚNICO USO DO CONJUNTO DE TESTE <<<")
print("=" * 78)

s_teste = escore(X_teste)
yhat_teste = (s_teste >= t_final).astype(int)
(tn_te, fp_te), (fn_te, tp_te) = matriz_confusao(y_teste, yhat_teste)

print(f"\nmatriz de confusão no teste (linhas = real, colunas = previsto):")
print(np.array([[tn_te, fp_te], [fn_te, tp_te]]))
print(f"\n{'':22s} {'validação':>12} {'TESTE':>12}")
m_val_f = avaliar_no_orcamento(y_val, escore(X_val))
for chave in ["recall", "precisão", "FPR", "% sinalizados"]:
    m_te = {"recall": tp_te / (tp_te + fn_te), "precisão": tp_te / (tp_te + fp_te),
            "FPR": fp_te / (fp_te + tn_te), "% sinalizados": yhat_teste.mean()}[chave]
    print(f"{chave:22s} {m_val_f[chave]:>12.4f} {m_te:>12.4f}")
print(f"{'AUC':22s} {m_val_f['AUC']:>12.4f} {roc_auc_score(y_teste, s_teste):>12.4f}")
print(f"{'AP':22s} {m_val_f['AP']:>12.4f} {average_precision_score(y_teste, s_teste):>12.4f}")
print(f"\ndefeitos encontrados: {tp_te} de {tp_te + fn_te} | módulos inspecionados: "
      f"{tp_te + fp_te} de {len(y_teste)} ({yhat_teste.mean():.1%}) | orçamento respeitado: "
      f"{yhat_teste.mean() <= ORCAMENTO}")

# ✅ RESPOSTA — Tarefa 6: custos no teste
c_FP_h, c_FN_h = 1.0, 10.0          # horas: 1 revisão desnecessária vs. 1 defeito que escapa
P_te, N_te = int(y_teste.sum()), int((y_teste == 0).sum())

politicas = pd.DataFrame([
    {"política": "não inspecionar nada", "FP": 0, "FN": P_te, "% inspecionado": 0.0},
    {"política": "inspecionar tudo", "FP": N_te, "FN": 0, "% inspecionado": 1.0},
    {"política": f"detector escolhido (t = {t_final:.3f})", "FP": fp_te, "FN": fn_te,
     "% inspecionado": yhat_teste.mean()},
]).set_index("política")
politicas["custo (horas)"] = c_FP_h * politicas["FP"] + c_FN_h * politicas["FN"]
print(f"\nCustos no teste com c_FP = {c_FP_h:.0f} h e c_FN = {c_FN_h:.0f} h "
      f"(n = {len(y_teste)}, P = {P_te}, N = {N_te}):")
display(politicas)
print(f"política de menor custo entre as três: {politicas['custo (horas)'].idxmin()}")

# Contraste: o limiar de custo de Elkan IGNORA o orçamento. Quanto a restrição custa?
t_elkan = c_FP_h / (c_FP_h + c_FN_h)
yhat_elkan = (s_teste >= t_elkan).astype(int)
(tn_e, fp_e), (fn_e, tp_e) = matriz_confusao(y_teste, yhat_elkan)
custo_elkan = c_FP_h * fp_e + c_FN_h * fn_e
print(f"\n(contraste) limiar de custo de Elkan t* = {t_elkan:.4f}:")
print(f"  FP={fp_e} FN={fn_e} -> custo = {custo_elkan:.0f} h, recall = {tp_e/(tp_e+fn_e):.4f}, "
      f"inspeciona {yhat_elkan.mean():.1%} da base")
print(f"  ou seja: a restrição de {ORCAMENTO:.0%} custa "
      f"{politicas['custo (horas)'].iloc[2] - custo_elkan:.0f} horas a mais que o ótimo "
      "de custo — e esse ótimo é inviável com a capacidade atual.")

> **Resposta do estudante — Tarefa 1 (métrica principal) e Tarefa 4 (decisão):**

---

## ✅ RESPOSTA — Tarefa 1 (métrica principal) e Tarefa 4 (decisão)

### Tarefa 1 — a métrica principal

**Métrica principal: recall no orçamento de 25%** — a fração dos módulos defeituosos que o detector encontra quando sinaliza, no máximo, 25% da base.

A justificativa vem do enunciado do problema, não do hábito. A gerência declarou **duas** coisas: uma **restrição** (capacidade de inspecionar 25% dos módulos) e um **objetivo** (encontrar o maior número possível de módulos defeituosos). Isso define uma otimização com restrição, e a métrica principal tem de ser o objetivo **avaliado dentro da restrição**. Qualquer outra escolha mediria outra coisa:

- **acurácia** é inútil aqui: o dummy que nunca alarma tem 0,846 e encontra zero defeitos (Seção 9);
- **$F_1$** embute a premissa de que precisão e recall valem igual — premissa que a gerência contradiz, já que um defeito que escapa de um sistema de solo de satélites custa muito mais que uma revisão desnecessária (Exercício 3, item 5);
- **AUC** avalia a ordenação em **todos** os limiares, inclusive regiões que nunca serão usadas (FPR > 0,8). Com orçamento fixo, a operação acontece num ponto só;
- **precisão** não serve como objetivo principal porque é trivialmente maximizada sinalizando pouquíssimos módulos — exatamente o detector tímido que o limiar 0,5 produz, com recall 0,18.

**Métricas secundárias reportadas:** precisão e FPR no ponto de operação (quanto esforço é desperdiçado), **fração sinalizada** (verificação de que o orçamento foi respeitado), **AUC** (qualidade da ordenação, independente do limiar, para o caso de o orçamento mudar na próxima release) e **AP** (resumo sensível à prevalência, mais informativo que a AUC em dados desbalanceados — Saito & Rehmsmeier, 2015). O **custo em horas** entra na Tarefa 6 como leitura financeira do mesmo ponto de operação.

**Referência obrigatória:** uma inspeção **aleatória** de 25% encontra, em média, 25% dos defeitos. É esse o número que qualquer candidato precisa bater para justificar sua existência.

### Tarefa 2 — comparação na validação (partição original)

| candidato | limiar | TP | FP | FN | recall | precisão | FPR | % sinaliz. | AUC | AP |
|:--|--:|--:|--:|--:|--:|--:|--:|--:|--:|--:|
| 1. baseline: ordenar por `loc` | 26,00 | 36 | 67 | 29 | 0,5538 | 0,3495 | 0,1877 | 24,4% | 0,8018 | 0,4048 |
| **2. LogReg (C=1)** | 0,2039 | 39 | 66 | 26 | 0,6000 | 0,3714 | 0,1849 | 24,9% | 0,8034 | 0,4378 |
| 3. LogReg balanced | 0,5951 | 40 | 65 | 25 | **0,6154** | **0,3810** | 0,1821 | 24,9% | 0,8041 | 0,4220 |
| 4. LogReg log1p | 0,2382 | 38 | 67 | 27 | 0,5846 | 0,3619 | 0,1877 | 24,9% | **0,8171** | 0,4607 |
| 5. LogReg C=0,1 | 0,2142 | 40 | 65 | 25 | **0,6154** | **0,3810** | 0,1821 | 24,9% | 0,8123 | **0,4661** |

Todos respeitam o orçamento. Todos batem a inspeção aleatória por uma margem grande: **0,55 a 0,62 contra 0,25**, ou seja, o detector encontra 2,2 a 2,5 vezes mais defeitos que sortear 25% dos módulos. A amplitude entre o melhor e o pior candidato é de apenas **0,0615** de recall — quatro defeitos de diferença em 65.

### Tarefa 3 — robustez (10 partições)

| candidato | média | desvio-padrão | mínimo | máximo |
|:--|--:|--:|--:|--:|
| 1. baseline: ordenar por `loc` | 0,5615 | 0,0637 | 0,4462 | 0,6615 |
| **2. LogReg (C=1)** | **0,6169** | 0,0686 | 0,5077 | 0,6923 |
| 3. LogReg balanced | 0,6092 | 0,0649 | 0,5231 | 0,6923 |
| 4. LogReg log1p | 0,6031 | 0,0640 | 0,4923 | 0,7231 |
| 5. LogReg C=0,1 | 0,6123 | 0,0664 | 0,5077 | 0,6923 |

**Este é o resultado decisivo da investigação.** A maior diferença entre as médias dos candidatos é **0,0554**, e o desvio-padrão típico **entre partições** é **0,0655** — ou seja, **a variabilidade é 1,2 vez maior que a maior diferença que eu estaria tentando explicar**. Em linguagem direta: *não há evidência de que qualquer um dos quatro candidatos aprendidos seja melhor que os outros*, e a vantagem sobre o baseline `loc` (≈ 5 pontos) é da mesma ordem do ruído. Note também que o candidato que ganhou na partição original (balanced/C=0,1, 0,6154) **não** é o de maior média nas 10 partições (LogReg C=1, 0,6169) — a troca de posição entre partições é a própria assinatura do empate estatístico.

### Tarefa 4 — decisão

**Detector escolhido: candidato 2, `StandardScaler` → `LogisticRegression(C=1.0)`, com limiar $t = 0{,}2039$ fixado na validação.**

Como a Tarefa 3 mostrou que as evidências **não distinguem** os candidatos, aplico critérios secundários explícitos, na ordem:

1. **calibração preservada.** O candidato 2 é o único cujos $\hat p$ continuam estimando $P(\text{defeito}\mid\mathbf x)$ na população real. O `balanced` desloca a média de $\hat p$ de 0,15 para ≈ 0,39 (Seção 15) e inviabiliza o limiar de custo de Elkan — e a Tarefa 6 depende exatamente disso. Essa é uma vantagem **funcional**, não estatística, e por isso decide onde o recall empata;
2. **simplicidade e custo de manutenção.** Duas etapas no `Pipeline`, um hiperparâmetro em valor padrão. O candidato 4 acrescenta uma transformação `log1p` que exige justificativa e documentação para ganhar 0,0 de recall; o 5 exige justificar por que $C = 0{,}1$;
3. **maior média entre as 10 partições** (0,6169) — critério de desempate fraco, usado por último justamente porque a diferença é menor que o desvio;
4. **o baseline `loc` fica de fora** por pouco: ele perde ≈ 5,5 pontos de recall em média (0,5615 contra 0,6169) e, principalmente, **não produz probabilidades**, o que o elimina de qualquer política baseada em custo. Registro, porém, que ele é um concorrente sério: se a equipe não tiver condições de manter um modelo em produção, ordenar por `loc` entrega ≈ 90% do benefício com ≈ 0% do custo de manutenção.

---

> **Conclusão técnica para a gerência de qualidade:**

---

## ✅ CONCLUSÃO TÉCNICA — Política de inspeção para a próxima release

**Política recomendada.** Adotar o detector de defeitos baseado em regressão logística sobre as 21 métricas estáticas (padronização + `LogisticRegression`, $C = 1$), operando com **limiar fixo de 0,204** na probabilidade estimada de defeito. Na prática, isso significa: extrair as métricas de cada módulo, calcular o escore, ordenar e **enviar para inspeção os módulos com escore ≥ 0,204** — cerca de 23% da base, dentro da capacidade de 25%. O limiar foi escolhido usando apenas treino e validação, e o conjunto de teste foi usado **uma única vez**, depois de todas as decisões, para produzir os números abaixo.

**O que esperar, com incerteza.** No conjunto de teste (422 módulos, 65 com defeito), a política encontrou **38 dos 65 módulos defeituosos (recall 0,585)** inspecionando **96 módulos (22,7% da base)**, dos quais 58 não tinham defeito (precisão 0,396; FPR 0,163). Em termos operacionais: **de cada 10 módulos defeituosos, a política aponta cerca de 6**, e **de cada 10 módulos enviados para revisão, cerca de 4 realmente têm defeito**. A incerteza não é desprezível e precisa ser comunicada junto do número: repetindo todo o procedimento em 10 partições aleatórias dos mesmos dados, o recall variou entre **0,51 e 0,69**, com média 0,617 e desvio-padrão **0,069**. A estimativa honesta é, portanto, **recall de 0,62 ± 0,07**, e o resultado do teste (0,585) está confortavelmente dentro dessa faixa — uma diferença de um único módulo em relação à validação (0,600). Para calibrar a expectativa: uma inspeção **aleatória** de 25% encontraria 25% dos defeitos. A política entrega **2,3 vezes mais** defeitos pelo mesmo esforço, e é esse o ganho real a ser creditado ao modelo.

**Esforço desperdiçado e custo.** Dos 96 módulos inspecionados no teste, **58 revisões (60%) não encontrarão defeito algum** — é o preço inevitável de classes sobrepostas, não um erro de implementação: módulos grandes e complexos sem defeito existem, e nenhum limiar os separa. Em horas, com 1 hora por revisão desnecessária e 10 horas por defeito que escapa até produção, as três políticas custam: **não inspecionar nada, 650 h**; **inspecionar tudo, 357 h**; **a política recomendada, 328 h** — a mais barata das três, com economia de 49% sobre não fazer nada e de 8% sobre inspecionar a frota inteira. Há, no entanto, um ponto que a gerência deve conhecer: o limiar que **minimiza o custo** ignorando a capacidade é $t^\star = c_{FP}/(c_{FP}+c_{FN}) = 0{,}091$, que custaria **239 h** (recall 0,846) mas exigiria inspecionar **46% da base**. A restrição de 25% nos custa, portanto, cerca de **89 horas por release**. Se o custo de um defeito em produção for maior que as 10 horas assumidas — plausível em um sistema de solo de satélites —, **ampliar a capacidade de inspeção é economicamente mais vantajoso que melhorar o modelo**, e essa é a decisão de maior impacto disponível hoje.

**Limitações que afetam estas estimativas** — em ordem de gravidade:

1. **A escolha do modelo não é sustentada por evidência.** Comparamos cinco candidatos (incluindo reponderação de classes, transformação logarítmica e outro valor de $C$) e a maior diferença entre suas médias foi 0,055, **menor** que o desvio-padrão de 0,066 entre partições. Nenhum é demonstravelmente superior; a escolha final se deu por calibração e simplicidade, não por desempenho. Em particular, **ordenar os módulos por linhas de código** — sem modelo algum — alcança recall médio de 0,56 e AUC de 0,802, contra 0,803 do modelo com 21 atributos. Se manter um modelo em produção se mostrar custoso, essa alternativa entrega quase o mesmo resultado sem nenhuma infraestrutura;
2. **Módulos duplicados inflam a avaliação.** Os 2.109 registros contêm apenas **1.192 vetores de atributos distintos**: módulos minúsculos (mediana de 4 linhas), quase todos sem defeito, repetem-se muitas vezes, e há vetores idênticos com rótulos **contraditórios**, o que impõe um piso de erro irredutível. Cópias do mesmo módulo podem cair em treino e teste, e a população avaliada é mais fácil que a real. Qualquer número aqui deve ser lido como **otimista**; a correção exige divisões que respeitem grupos de módulos idênticos;
3. **A precisão não é transportável para outro projeto.** Precisão e custo dependem da **prevalência**: o KC1 tem 15,4% de módulos defeituosos. Pela identidade de Bayes (Seção 12.6), o **mesmo** detector aplicado a um projeto com 5% de defeitos teria precisão de cerca de 0,17 em vez de 0,40 — mais que o dobro de revisões desperdiçadas por defeito encontrado — enquanto recall e FPR permaneceriam os mesmos. **Antes de reutilizar esta política em outro sistema, é obrigatório recalibrar o limiar na prevalência local**;
4. **Métricas estáticas medem tamanho, não qualidade.** O resultado do Exercício 4 sugere que a maior parte do poder preditivo se reduz a "módulos maiores têm mais defeitos". A política é útil como **priorização de esforço**, não como diagnóstico: ela não diz *onde* está o defeito nem *por que*, e não substitui revisão por especialista;
5. **O resultado vale para a release medida.** Mudanças de equipe, de linguagem, de padrão de codificação ou de processo de registro de defeitos invalidam o modelo. Recomendo revalidar a cada release, comparando o recall observado com a faixa 0,62 ± 0,07 e retreinando quando ele cair fora dela por duas releases consecutivas.

**Recomendação final em uma linha:** adotar a política com limiar 0,204, reportar sempre **recall ± desvio** em vez de um número isolado, medir o recall efetivo da próxima release e, em paralelo, avaliar o retorno de ampliar a capacidade de inspeção de 25% para ~45% — que, pelos custos informados, vale mais que qualquer ganho de modelagem disponível hoje.

---

---
# Referências

**Bibliografia da disciplina**

- FACELI, K.; LORENA, A. C.; GAMA, J.; CARVALHO, A. C. P. L. F. de; ALMEIDA, T. A. de. *Inteligência artificial: uma abordagem de aprendizado de máquina*. Rio de Janeiro: LTC, 2025. (Seção sobre métricas de avaliação de classificadores.)
- RUSSELL, S. J.; NORVIG, P. *Inteligência artificial: uma abordagem moderna*. Rio de Janeiro: GEN LTC, 2022. (Cap. 12, probabilidade e regra de Bayes; cap. 19, aprendizado a partir de exemplos.)

**Artigos e livros citados**

- BUITINCK, L. et al. API design for machine learning software: experiences from the scikit-learn project. *ECML PKDD Workshop: Languages for Data Mining and Machine Learning*, 2013. arXiv:1309.0238.
- DAVIS, J.; GOADRICH, M. The relationship between Precision-Recall and ROC curves. In: *Proceedings of the 23rd International Conference on Machine Learning (ICML)*, p. 233–240, 2006. DOI: 10.1145/1143844.1143874.
- ELKAN, C. The foundations of cost-sensitive learning. In: *Proceedings of the 17th International Joint Conference on Artificial Intelligence (IJCAI)*, p. 973–978, 2001.
- FAWCETT, T. An introduction to ROC analysis. *Pattern Recognition Letters*, v. 27, n. 8, p. 861–874, 2006. DOI: 10.1016/j.patrec.2005.10.010.
- HALSTEAD, M. H. *Elements of Software Science*. New York: Elsevier North-Holland, 1977.
- HANLEY, J. A.; McNEIL, B. J. The meaning and use of the area under a receiver operating characteristic (ROC) curve. *Radiology*, v. 143, n. 1, p. 29–36, 1982. DOI: 10.1148/radiology.143.1.7063747.
- LIU, D. C.; NOCEDAL, J. On the limited memory BFGS method for large scale optimization. *Mathematical Programming*, v. 45, p. 503–528, 1989. DOI: 10.1007/BF01589116.
- McCABE, T. J. A complexity measure. *IEEE Transactions on Software Engineering*, v. SE-2, n. 4, p. 308–320, 1976. DOI: 10.1109/TSE.1976.233837.
- MENZIES, T.; GREENWALD, J.; FRANK, A. Data mining static code attributes to learn defect predictors. *IEEE Transactions on Software Engineering*, v. 33, n. 1, p. 2–13, 2007. DOI: 10.1109/TSE.2007.256941.
- SAITO, T.; REHMSMEIER, M. The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLoS ONE*, v. 10, n. 3, e0118432, 2015. DOI: 10.1371/journal.pone.0118432.
- SHEPPERD, M.; SONG, Q.; SUN, Z.; MAIR, C. Data quality: some comments on the NASA software defect datasets. *IEEE Transactions on Software Engineering*, v. 39, n. 9, p. 1208–1215, 2013. DOI: 10.1109/TSE.2013.11.
- VAN RIJSBERGEN, C. J. *Information Retrieval*. 2. ed. London: Butterworths, 1979.

**Dados**

- SAYYAD SHIRABAD, J.; MENZIES, T. J. *The PROMISE Repository of Software Engineering Databases*. School of Information Technology and Engineering, University of Ottawa, Canada, 2005. Conjunto KC1 (NASA Metrics Data Program).
- VANSCHOREN, J.; VAN RIJN, J. N.; BISCHL, B.; TORGO, L. OpenML: networked science in machine learning. *SIGKDD Explorations*, v. 15, n. 2, p. 49–60, 2014. DOI: 10.1145/2641190.2641198. KC1: https://www.openml.org/d/1067

**Documentação de software**

- PEDREGOSA, F. et al. Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research*, v. 12, p. 2825–2830, 2011.
- scikit-learn (2026). *User Guide* (versão estável): *Logistic regression* — https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression ; *Metrics and scoring* — https://scikit-learn.org/stable/modules/model_evaluation.html ; *Tuning the decision threshold for class prediction* — https://scikit-learn.org/stable/modules/classification_threshold.html ; `fetch_openml` — https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_openml.html
- HARRIS, C. R. et al. Array programming with NumPy. *Nature*, v. 585, p. 357–362, 2020. DOI: 10.1038/s41586-020-2649-2.
- McKINNEY, W. Data structures for statistical computing in Python. In: *Proceedings of the 9th Python in Science Conference*, p. 56–61, 2010. DOI: 10.25080/Majora-92bf1922-00a.
- HUNTER, J. D. Matplotlib: a 2D graphics environment. *Computing in Science & Engineering*, v. 9, n. 3, p. 90–95, 2007. DOI: 10.1109/MCSE.2007.55.